# 공정 조합 퍼널 · 실제 데이터 분석

설비 이력 CSV(`raw.csv`)를 읽어서, 스텝 안의 오더별 유닛 조합 중 불량률을 끌어올린 **대표 불량 대상**을 찾고 랭킹과 차트로 보여 줍니다. 판정 로직은 main 브랜치의 웹 화면, `funnel_check.ipynb`와 같습니다.

**데이터 규칙**
- 웨이퍼 = `root_lot_id` + `wafer_id`. 웨이퍼마다 `y_value`와 `good_bad`(G/B)가 하나씩 있습니다. 판정은 두 값을 함께 봅니다.
- 스텝 = `step_seq`에서 끝의 `_번호`를 뗀 값(`step_num`), 오더 = 그 번호. 예: `ex100000_1` ~ `ex100000_6` → 스텝 `ex100000` 안의 오더 1~6. 끝 번호가 없는 `step_seq`는 그대로 스텝으로 보고 `step_ord`를 오더로 씁니다.
- 유닛 = `eqp_id` + `-` + `chamber_id` (chamber가 없으면 `eqp_id`)

| 절 | 내용 |
|---|---|
| 0 | 설정 |
| 1 | 데이터 읽기와 점검: job · 제품 필터, 웨이퍼 키, 값 충돌, 결측, 재작업 중복 |
| 2 | 분석용 구조로 바꾸기: 스텝 · 오더 · 유닛 |
| 3 | 분석 · 차트 함수 (수정할 필요 없음) |
| 4 | 자동 점검(분포 모양 · part · 랏 차이)과 조합 판정 |
| 5 | 한 장의 funnel과 대표 불량 대상 랭킹 |
| 6 | 대표 불량 대상 자세히 보기: 후보, 스텝 흐름, 웨이퍼 분포, 왜 대표인가, 시간 추이 |
| 7 | 스텝 간 설비 연관 점검 |

## 0. 설정
데이터는 `unit_combi` 폴더(저장소 최상위)의 **`raw.csv` 하나만** 읽습니다. 실제 데이터를 이 이름으로 두세요. 저장소에는 가상 demo 데이터 `demo_raw.csv`만 있어서, demo로 확인하려면 `raw.csv`로 복사해 실행합니다. 아래 값만 바꿔서 위에서부터 다시 실행하면 됩니다.

In [ ]:
JOB_ID = None         # 여러 job이 섞여 있으면 지정. None이면 analysis_date가 가장 최근인 job
STEP_SUFFIX = r'_(\d+)$'  # step_seq 끝의 오더 번호: ex100000_3 → 스텝 ex100000 · 오더 3. None이면 step_seq를 스텝, step_ord를 오더로
PART_ID = None         # 보통 None: 스텝 기준으로 모든 part를 함께 분석. 한 제품만 볼 때만 지정
LINE_ID = None         # 보통 None. 한 라인만 볼 때만 지정
HIGHER_IS_WORSE = True # y_value가 클수록 나쁘면 True. 수율처럼 클수록 좋으면 False (부호를 뒤집어 분석)
Y_TRANSFORM = 'auto'   # 한쪽으로 길게 치우친 y_value: 'auto' 왜도가 1보다 크면 순위 점수 · 'rank' 순위 점수 · 'log' 로그 · None 그대로
PART_ADJUST = 'auto'   # part별 y_value · bad 비율 수준 차이 보정: 'auto' 차이가 뚜렷하면 보정 · True 항상 · False 안 함
REWORK = 'last'        # 같은 웨이퍼 · 스텝 · 오더 이력이 여러 줄이면: 'last' 마지막 track-in · 'first' 처음 track-in
MIN_N = 20             # 조합 최소 웨이퍼 수. 웨이퍼가 1,000장보다 적으면 10 정도로 낮춰 볼 것
MAX_DEPTH = 3          # 조합에 묶을 최대 오더 수 N. 오더 1개 ~ N개 조합을 본다 (크게 할수록 계산이 늘어남)
SAME_WAFERS = 0.9      # 웨이퍼가 이만큼 겹치는 대표 대상(같은 챔버에서 도는 하위 스텝 등)은 한 줄로 묶음. None이면 묶지 않음
PICK = 1               # 6절에서 자세히 볼 대표 대상의 순위
STEP = None            # 특정 스텝만 볼 때 스텝 값 (예: 'ex100000'). None이면 전체
FLOW_MAX_ORDERS = 12   # 스텝 흐름 차트에 그릴 최대 오더 수 (경로 · 유닛 간 차이가 있는 오더 우선)
LOGX = True            # funnel 가로축 로그
SAVE_RANKING = None    # 'ranking.csv'처럼 주면 랭킹 표를 CSV로 저장
FONT_PATH = None       # 따로 쓸 한글 글꼴(.ttf) 경로. None이면 노트북 옆 fonts 폴더의 나눔고딕
SEED = 0               # 그림 흔들기용 난수 시드

In [ ]:
import math
import re
import sys
import time
from collections import defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch, PathPatch, Rectangle
from matplotlib.path import Path as MplPath
from matplotlib.ticker import FuncFormatter, LogLocator, NullLocator

assert sys.version_info >= (3, 12), f'Python 3.12 이상이 필요합니다 (현재 {sys.version.split()[0]})'
Y_NAME = 'y_value' if HIGHER_IS_WORSE else '−y_value'

C = {
    'surface': '#fcfcfb', 'ink': '#0b0b0b', 'ink2': '#52514e', 'muted': '#898781',
    'grid': '#e1e0d9', 'axis': '#c3c2b7', 'band': '#ecebe6',
    'bad': '#d03b3b', 'accent': '#2a78d6', 'mid': '#f0efec',
}
# 한글 글꼴: FONT_PATH → 노트북 옆 fonts 폴더(저장소에 든 나눔고딕, SIL OFL) → 시스템 글꼴 순으로 찾는다
font_files = [Path(FONT_PATH)] if FONT_PATH else [
    p for d in (Path.cwd() / 'fonts', Path.cwd() / 'notebooks' / 'fonts') if d.is_dir() for p in sorted(d.glob('*.tt[fc]'))]
for p in font_files:
    mpl.font_manager.fontManager.addfont(str(p))
if font_files:
    KR_FONT = mpl.font_manager.FontProperties(fname=str(font_files[0])).get_name()
else:
    _fonts = {f.name for f in mpl.font_manager.fontManager.ttflist}
    KR_FONT = next((f for f in ('NanumGothic', 'Nanum Gothic', 'Noto Sans CJK KR', 'Noto Sans KR', 'Malgun Gothic',
                                'Apple SD Gothic Neo', 'AppleGothic') if f in _fonts), None)
if KR_FONT is None:
    print('⚠ 한글 글꼴을 찾지 못해 차트의 한글이 □로 보일 수 있습니다. 저장소의 notebooks/fonts 폴더를 노트북 옆에 두거나 FONT_PATH를 지정하세요.')
plt.rcParams.update({
    'font.family': [KR_FONT or 'DejaVu Sans', 'DejaVu Sans'], 'axes.unicode_minus': False, 'font.size': 10,
    'figure.dpi': 100, 'figure.facecolor': C['surface'], 'axes.facecolor': C['surface'], 'savefig.facecolor': C['surface'],
    'axes.edgecolor': C['axis'], 'axes.linewidth': 0.8, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': C['grid'], 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'axes.titlecolor': C['ink'], 'axes.titlepad': 10,
    'axes.labelcolor': C['ink2'], 'axes.labelsize': 10,
    'xtick.color': C['axis'], 'ytick.color': C['axis'], 'xtick.labelcolor': C['ink2'], 'ytick.labelcolor': C['ink2'],
    'xtick.major.size': 0, 'ytick.major.size': 0, 'xtick.minor.size': 0, 'ytick.minor.size': 0,
    'legend.frameon': False, 'legend.fontsize': 9,
})
pd.set_option('display.max_rows', 80)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', '{:,.4g}'.format)
print(f'Python {sys.version.split()[0]} · numpy {np.__version__} · pandas {pd.__version__} · matplotlib {mpl.__version__} · 글꼴 {KR_FONT}')

## 1. 데이터 읽기와 점검
- 컬럼을 확인하고, job · 제품 · 라인으로 좁힙니다. job이 여러 개면 가장 최근 분석을 씁니다.
- 웨이퍼마다 `y_value`, `good_bad`가 하나인지 확인합니다. 값이 둘 이상이면 첫 값을 쓰고 개수를 알립니다.
- `step_seq` 끝의 `_번호`를 떼어 스텝으로 묶고, 그 번호를 오더로 씁니다. 묶인 모습은 오더가 많은 스텝부터 표로 보여 줍니다.
- 같은 웨이퍼 · 스텝 · 오더 이력이 여러 줄이면(재작업 등) `REWORK` 규칙으로 하나만 남깁니다.

In [ ]:
REQUIRED = ['analysis_date', 'job_id', 'root_lot_id', 'lot_id', 'wafer_id', 'tkin_time', 'line_id', 'part_id',
            'step_seq', 'step_desc', 'eqp_id', 'chamber_id', 'ppid', 'y_value', 'good_bad', 'step_ord']
USE = [c for c in REQUIRED if c not in ('lot_id', 'ppid')]          # 분석에 쓰는 컬럼만 읽는다
BAD_TOKENS = {'b', 'bad', 'ng', 'n/g', 'fail', 'f', '1', 'true', 'y', 'yes', '불량'}
GOOD_TOKENS = {'g', 'good', 'ok', 'pass', 'p', '0', 'false', 'n', 'no', '양품'}


def ord_label(o):
    return str(int(o)) if float(o).is_integer() else str(o)


def cat_values(s, f, fill=np.nan):
    """category 컬럼의 고유값에만 f를 적용해 줄마다의 값 배열로 (결측은 fill). 수백만 줄도 고유값 수만큼만 계산한다"""
    vals = np.asarray(f(pd.Series(s.cat.categories, dtype=object)))
    return np.concatenate([vals, np.array([fill]).astype(vals.dtype)])[s.cat.codes.to_numpy()]


def cat_recode(s, f):
    """category 컬럼의 고유값에 f를 적용한 새 category (바꾼 값이 같아지면 하나로 합친다)"""
    new = np.asarray(f(pd.Series(s.cat.categories, dtype=object)), dtype=object)
    ok = pd.notna(new)
    u, inv = np.unique(new[ok].astype(str), return_inverse=True)
    code = np.full(len(new) + 1, -1)
    code[np.flatnonzero(ok)] = inv
    return pd.Categorical.from_codes(code[s.cat.codes.to_numpy()], u)


def norm_wid(v):
    """wafer_id가 숫자면 '01'과 '1'을 같게 본다"""
    num = pd.to_numeric(v, errors='coerce')
    is_int = num.notna() & (num == num.round())
    out = v.copy()
    out[is_int] = num[is_int].astype(np.int64).astype(str)
    return out


t0 = time.perf_counter()
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_PATH = REPO / 'raw.csv'               # unit_combi 폴더의 raw.csv 하나만 읽는다
assert CSV_PATH.exists(), (f'raw.csv가 없습니다: {CSV_PATH}\n실제 데이터를 이 이름으로 두세요. '
                           f'demo로 확인하려면 같은 폴더의 demo_raw.csv를 raw.csv로 복사하세요.')
print(f'읽는 파일: {CSV_PATH.resolve()}')
miss_cols = [c for c in REQUIRED if c not in pd.read_csv(CSV_PATH, nrows=0).columns]
assert not miss_cols, f'CSV에 없는 컬럼: {miss_cols}'
# 컬럼마다 값 종류가 줄 수보다 훨씬 적어서 category로 읽으면 메모리가 약 10분의 1로 줄고, 공백 정리 · 변환도 고유값에만 하면 된다.
# tkin_time은 줄마다 거의 달라서 문자열로 읽어 바로 시각으로 바꾼다
raw = pd.read_csv(CSV_PATH, usecols=USE, dtype={**{c: 'category' for c in USE}, 'tkin_time': str})
raw['t'] = pd.to_datetime(raw.pop('tkin_time'), errors='coerce')
for c in USE:
    if c != 'tkin_time':
        raw[c] = cat_recode(raw[c], lambda v: v.str.strip())
raw['analysis_date'] = raw['analysis_date'].cat.as_ordered()
n_read = len(raw)
print(f'읽기 {time.perf_counter() - t0:.1f}초 · {n_read:,}줄')

# 웨이퍼 키: root_lot_id와 wafer_id 코드의 쌍 (정수)
wid = cat_recode(raw['wafer_id'], norm_wid)
root_code, wid_code = raw['root_lot_id'].cat.codes.to_numpy().astype(np.int64), wid.codes.astype(np.int64)
no_id = (root_code < 0) | (wid_code < 0)
raw['wafer_key'] = root_code * (len(wid.categories) + 1) + wid_code
n_no_id = int(no_id.sum())
raw = raw[~no_id]

jobs = (raw.groupby('job_id', dropna=False, observed=True)
        .agg(analysis_date=('analysis_date', 'max'), 줄=('job_id', 'size'), 웨이퍼=('wafer_key', 'nunique'))
        .sort_values('analysis_date'))
if len(jobs) > 1:
    if JOB_ID is None:
        JOB_ID = jobs.index[-1]
    print(f'⚠ job이 {len(jobs)}개 섞여 있어 {JOB_ID}만 씁니다 (JOB_ID로 바꿀 수 있음)')
    display(jobs)
for col, val in (('job_id', JOB_ID), ('part_id', PART_ID), ('line_id', LINE_ID)):
    if val is not None:
        raw = raw[raw[col] == val]
print(f"part_id {raw['part_id'].nunique()}개 · line_id {raw['line_id'].nunique()}개를 스텝 기준으로 함께 분석합니다. "
      "part별 y_value 수준 차이는 4절에서 점검합니다.")

to_num = lambda v: pd.to_numeric(v, errors='coerce').to_numpy(float)
raw['y'] = cat_values(raw['y_value'], to_num)
# 스텝과 오더: step_seq 끝의 '_번호'를 떼어 스텝(step_num)으로 묶고 그 번호를 오더로 쓴다. 끝 번호가 없으면 step_seq 그대로 · step_ord
if STEP_SUFFIX:
    sfx = cat_values(raw['step_seq'], lambda v: to_num(v.str.extract(STEP_SUFFIX, expand=False)))
    raw['step'] = cat_recode(raw['step_seq'], lambda v: v.str.replace(STEP_SUFFIX, '', regex=True))
else:
    sfx = np.full(len(raw), np.nan)
    raw['step'] = raw['step_seq']
step_ord = cat_values(raw['step_ord'], to_num)
raw['ord'] = np.where(np.isnan(sfx), step_ord, sfx)
n_ord_diff = int((~np.isnan(sfx) & ~np.isnan(step_ord) & (sfx != step_ord)).sum())
raw['bad'] =cat_values(raw['good_bad'], lambda v: np.where(v.str.lower().isin(BAD_TOKENS), 1.0,
                                                            np.where(v.str.lower().isin(GOOD_TOKENS), 0.0, np.nan)))
unknown_gb = raw.loc[raw['good_bad'].notna() & raw['bad'].isna(), 'good_bad'].value_counts()
unknown_gb = unknown_gb[unknown_gb > 0]

# 웨이퍼 표: 웨이퍼마다 값 하나
wf = raw.groupby('wafer_key').agg(root_lot_id=('root_lot_id', 'first'), part=('part_id', 'first'), part_n=('part_id', 'nunique'),
                                  y=('y', 'first'), y_n=('y', 'nunique'), bad=('bad', 'first'), bad_n=('bad', 'nunique'))
wf['root_lot_id'], wf['part'] = wf['root_lot_id'].astype(object), wf['part'].astype(object)
has_bad = bool(wf['bad'].notna().any())                 # good_bad가 하나도 없으면 y_value만으로 판정
keep = wf['y'].notna() & (wf['bad'].notna() if has_bad else True)

# 이력 표: 오더 · 설비가 있는 줄만
h = raw.loc[raw['wafer_key'].isin(wf.index[keep]) & raw['ord'].notna() & raw['eqp_id'].notna() & raw['step'].notna(),
            ['wafer_key', 'step', 'step_seq', 'step_desc', 'ord', 'eqp_id', 'chamber_id', 't']].copy()
# 유닛 = eqp_id-chamber_id (chamber가 없으면 eqp_id). 설비 · 챔버 코드 쌍에서 고유한 쌍에만 이름을 붙인다
e_cat, c_cat = h['eqp_id'].cat.categories, h['chamber_id'].cat.categories
pair = h['eqp_id'].cat.codes.to_numpy().astype(np.int64) * (len(c_cat) + 1) + h['chamber_id'].cat.codes.to_numpy() + 1
pairs, inv = np.unique(pair, return_inverse=True)
names = np.array([e_cat[p // (len(c_cat) + 1)] + ('-' + c_cat[p % (len(c_cat) + 1) - 1] if p % (len(c_cat) + 1) and c_cat[p % (len(c_cat) + 1) - 1] else '')
                  for p in pairs], dtype=object)
u_names, u_inv = np.unique(names, return_inverse=True)
h['unit'] = pd.Categorical.from_codes(u_inv[inv], u_names)
n_dup_keys = int(h.duplicated(['wafer_key', 'step', 'ord'], keep='first').sum())
h = (h.sort_values('t', na_position='first' if REWORK == 'last' else 'last', kind='stable')
      .drop_duplicates(['wafer_key', 'step', 'ord'], keep=REWORK))

quality = pd.DataFrame([
    ('읽은 줄', n_read),
    ('root_lot_id · wafer_id가 없어 뺀 줄', n_no_id),
    ('필터 후 줄', len(raw)),
    ('웨이퍼 (root_lot_id + wafer_id)', len(wf)),
    ('y_value가 없어 뺀 웨이퍼', int(wf['y'].isna().sum())),
    ('good_bad가 없어 뺀 웨이퍼 (y_value는 있음)', int((wf['y'].notna() & wf['bad'].isna()).sum()) if has_bad else 0),
    ('y_value가 둘 이상인 웨이퍼 (첫 값 사용)', int((wf['y_n'] > 1).sum())),
    ('good_bad가 둘 이상인 웨이퍼 (첫 값 사용)', int((wf['bad_n'] > 1).sum())),
    ('part_id가 둘 이상인 웨이퍼 (첫 값 사용)', int((wf['part_n'] > 1).sum())),
    ('good_bad를 알 수 없는 줄', int(unknown_gb.sum())),
    ('오더 번호 · eqp_id · step_seq가 없어 뺀 줄', int((raw['ord'].isna() | raw['eqp_id'].isna() | raw['step'].isna()).sum())),
    ('step_ord가 step_seq 끝 번호와 다른 줄 (끝 번호 사용)', n_ord_diff),
    ('tkin_time을 읽지 못한 줄', int(raw['t'].isna().sum())),
    (f"같은 웨이퍼 · 스텝 · 오더가 겹친 줄 ('{REWORK}'만 남김)", n_dup_keys),
    ('분석에 쓰는 이력 줄', len(h)),
], columns=['항목', '값']).set_index('항목')
display(quality)
if len(unknown_gb):
    print('알 수 없는 good_bad 값:', dict(unknown_gb.head(10)))
if n_ord_diff:
    print(f'⚠ step_ord가 step_seq 끝 번호와 다른 줄이 {n_ord_diff:,}개 있습니다. 오더는 step_seq 끝 번호를 씁니다.')

# 스텝으로 묶인 모습: 오더가 많은 스텝부터
so = h.drop_duplicates(['step', 'ord']).sort_values('ord')
step_map = so.groupby('step').agg(
    오더=('ord', 'size'),
    오더_번호=('ord', lambda s: ', '.join(map(ord_label, s)) if len(s) <= 8 else f'{ord_label(s.iat[0])} … {ord_label(s.iat[-1])}'),
    step_seq=('step_seq', lambda s: s.iat[0] if len(s) == 1 else f'{s.iat[0]} … {s.iat[-1]}'),
    step_desc=('step_desc', lambda s: ' / '.join(list(dict.fromkeys(s.dropna()))[:3]) + (' …' if s.nunique() > 3 else '')))
step_map = step_map.sort_values('오더', ascending=False, kind='stable').rename(columns={'오더': '오더 수', '오더_번호': '오더 번호'})
print(f"step_seq {so['step_seq'].nunique():,}개 → 스텝 {len(step_map):,}개 · 스텝당 오더 최대 {step_map['오더 수'].max()}개")
display(step_map.head(8).rename_axis('스텝'))

## 2. 분석용 구조로 바꾸기
스텝마다 오더와 유닛 목록을 만들고, 웨이퍼마다 오더별로 어느 유닛을 지났는지 배열로 정리합니다. 유닛 이름에는 오더가 없어서, 경로는 `O2:EQP01-A → O3:EQP07-B`처럼 오더를 붙여 씁니다. 스텝 `ex100000`의 `O2`는 `step_seq` `ex100000_2`입니다.

In [ ]:
MISSING = 0xFFFF


def step_sort_key(s):
    return (0, float(s), s) if s.replace('.', '', 1).isdigit() else (1, 0.0, s)


def group_mode(g, codes, cats, G):
    """묶음 g(0..G-1)마다 가장 흔한 category 값 (개수가 같으면 이름순으로 앞선 값)"""
    out = np.full(G, '', dtype=object)
    ok = codes >= 0
    if ok.any():
        pk, cnt = np.unique(g[ok] * len(cats) + codes[ok], return_counts=True)
        gg, cc = pk // len(cats), pk % len(cats)
        o = np.lexsort((cc, -cnt, gg))
        first = o[np.r_[True, gg[o][1:] != gg[o][:-1]]]
        out[gg[first]] = np.asarray(cats, dtype=object)[cc[first]]
    return out


t0 = time.perf_counter()
wafers = wf.index[keep]
N = len(wafers)
# 줄마다 (스텝 순위, 오더, 유닛)으로 한 번 정렬하고 경계만 찾아서, 스텝 · 오더마다 따로 거르지 않고 배열을 채운다
code_of = lambda col: h[col].cat.codes.to_numpy().astype(np.int64)
step_names = np.asarray(h['step'].cat.categories, dtype=object)
order_ = sorted(np.unique(code_of('step')), key=lambda c: step_sort_key(step_names[c]))
rank = np.full(len(step_names), -1)
rank[order_] = np.arange(len(order_))
s_arr, o_arr, u_arr = rank[code_of('step')], h['ord'].to_numpy(float), code_of('unit')
w_arr = pd.Series(np.arange(N), index=wafers).reindex(h['wafer_key'].to_numpy()).to_numpy().astype(np.int64)
srt = np.lexsort((u_arr, o_arr, s_arr))
s_arr, o_arr, u_arr, w_arr = s_arr[srt], o_arr[srt], u_arr[srt], w_arr[srt]
t_arr = h['t'].to_numpy('datetime64[ns]')[srt]
new_g = np.r_[True, (s_arr[1:] != s_arr[:-1]) | (o_arr[1:] != o_arr[:-1])]
g_arr = np.cumsum(new_g) - 1                            # (스텝, 오더) 묶음 번호
G, g_start = int(g_arr[-1]) + 1, np.flatnonzero(new_g)
new_pu = new_g | np.r_[True, u_arr[1:] != u_arr[:-1]]
pu = np.cumsum(new_pu) - 1                              # (스텝, 오더, 유닛) 고유 쌍 번호
local = pu - pu[g_start][g_arr]                         # 오더 안 유닛 번호 (유닛 이름순)
assert local.max() < MISSING, '한 오더의 유닛이 너무 많습니다'
pu_start = np.flatnonzero(new_pu)
units_of = np.split(np.asarray(h['unit'].cat.categories, dtype=object)[u_arr[pu_start]], np.flatnonzero(np.diff(g_arr[pu_start])) + 1)
seq_mode = group_mode(g_arr, code_of('step_seq')[srt], h['step_seq'].cat.categories, G)
desc_mode = group_mode(g_arr, code_of('step_desc')[srt], h['step_desc'].cat.categories, G)
proc_mode = group_mode(s_arr, code_of('step_desc')[srt], h['step_desc'].cat.categories, len(order_))
t_int = np.where(np.isnat(t_arr), np.iinfo(np.int64).max, t_arr.view(np.int64))
bounds = np.r_[np.flatnonzero(np.r_[True, s_arr[1:] != s_arr[:-1]]), len(s_arr)]
steps, assign, trackin = [], [], []
for s_i, (r0, r1) in enumerate(zip(bounds[:-1], bounds[1:])):
    q0, q1 = g_arr[r0], g_arr[r1 - 1] + 1
    A = np.full((q1 - q0, N), MISSING, dtype=np.uint16)
    A[g_arr[r0:r1] - q0, w_arr[r0:r1]] = local[r0:r1]
    first_t = np.full(N, np.iinfo(np.int64).max)
    np.minimum.at(first_t, w_arr[r0:r1], t_int[r0:r1])   # 웨이퍼마다 이 스텝의 첫 track-in
    tk = first_t.view('datetime64[ns]').copy()
    tk[first_t == np.iinfo(np.int64).max] = np.datetime64('NaT', 'ns')
    seqs = [{'name': f'O{ord_label(o_arr[g_start[g]])}', 'units': list(units_of[g]), 'seq': seq_mode[g], 'desc': desc_mode[g]}
            for g in range(q0, q1)]
    steps.append({'index': s_i, 'name': str(step_names[order_[s_i]]), 'proc': proc_mode[s_i], 'seqs': seqs})
    assign.append(A)
    trackin.append(tk)

Y = wf.loc[wafers, 'y'].to_numpy(float)
if not HIGHER_IS_WORSE:
    Y = -Y                                     # 작을수록 나쁜 값은 부호를 뒤집어 '클수록 나쁨'으로 맞춘다
assert N >= 2 * MIN_N and Y.std() > 0, f'비교할 수 없습니다: 웨이퍼 {N}장, y_value 표준편차 {Y.std():.3g}'
B = wf.loc[wafers, 'bad'].to_numpy(float) if has_bad else None
if B is None:
    print('⚠ good_bad가 없어 y_value만으로 판정합니다.')
elif B.std() == 0:
    print(f'⚠ good_bad가 한쪽뿐입니다 (bad 비율 {B.mean():.0%}). 양품 · 불량 웨이퍼가 함께 있어야 비교할 수 있어서 y_value만으로 판정합니다.')
    B = None
lots = wf.loc[wafers, 'root_lot_id'].to_numpy()
lot_idx, lot_names = pd.factorize(lots)
data = {'N': N, 'Y': Y, 'B': B, 'steps': steps, 'assign': assign, 'trackin': trackin, 'wafers': wafers,
        'lots': lots, 'lot_idx': lot_idx, 'n_lots': len(lot_names), 'parts': wf.loc[wafers, 'part'].fillna('').to_numpy(),
        'bad': wf.loc[wafers, 'bad'].to_numpy(float),
        'value': wf.loc[wafers, 'y'].to_numpy(float)}
n_orders = sum(len(s['seqs']) for s in steps)
print(f"변환 {time.perf_counter() - t0:.1f}초 · 웨이퍼 {N:,}장 · 랏 {len(set(data['lots'])):,}개 · 스텝 {len(steps)}개 · "
      f"오더 {n_orders}개 · 유닛 {sum(len(q['units']) for s in steps for q in s['seqs']):,}개")
print(f"{Y_NAME} 평균 {Y.mean():.4g} · 표준편차 {Y.std(ddof=1):.4g}"
      + f" · 왜도 {pd.Series(Y).skew():.2f}"
      + (f" · bad 비율 {np.nanmean(data['bad']):.1%}" if np.isfinite(data['bad']).any() else ''))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
cnt = np.bincount([len(s['seqs']) for s in steps])
axes[0].bar(np.arange(1, len(cnt)), cnt[1:], width=0.6, color=C['accent'])
axes[0].set(title='스텝별 오더 수', xlabel='오더 수', ylabel='스텝 수')
cnt = np.bincount([len(q['units']) for s in steps for q in s['seqs']])
axes[1].bar(np.arange(1, len(cnt)), cnt[1:], width=0.6, color=C['accent'])
axes[1].set(title='오더별 유닛 수', xlabel='유닛 수', ylabel='오더 수')
v, b = data['value'], data['bad']
if np.isfinite(b).any() and np.isfinite(v).any():
    bins = np.linspace(np.nanmin(v), np.nanmax(v), 50)
    axes[2].hist([v[b == 0], v[b == 1]], bins=bins, stacked=True, color=[C['muted'], C['bad']], label=['good', 'bad'],
                 edgecolor=C['surface'], linewidth=0.5)
    axes[2].legend()
else:
    axes[2].hist(v[np.isfinite(v)], bins=50, color=C['muted'])
axes[2].set(title='웨이퍼 y_value 분포', xlabel='y_value', ylabel='웨이퍼 수')
for ax in axes:
    ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()

## 3. 분석 · 차트 함수
`funnel_check.ipynb`에서 웹 JS와 값 단위로 맞춰 본 로직을 그대로 옮겼습니다. 수정할 필요 없이 실행만 하면 됩니다.
- **조합**: 한 스텝 안에서 오더 1~`MAX_DEPTH`개(N개)를 골라, 고른 오더마다 정해진 유닛을 모두 지난 웨이퍼 묶음. 오더는 이웃하지 않아도 됩니다(예: `O3 → O17`). 웨이퍼 `MIN_N`장 이상만 봅니다. 오더 3개 이상 조합은 오더 하나를 뺀 부모 조합 중 하나가 z 1.96을 넘을 때만 계산합니다(계산량을 줄이려고).
- **기준 z**: 오더 수별로 가능한 조합 수만큼 보정합니다(Bonferroni).
- **z**: 조합마다 y_value 차이의 z와 bad 비율 차이의 z를 각각 랏 랜덤 효과 모형으로 구해 하나로 합칩니다(4절 ③). 같은 랏 안의 비교는 랏 차이가 상쇄되고, 랏이 통째로 지난 유닛은 랏 단위로 비교합니다. 부모 비교 · 하위 기인 · 스텝 간 판정도 같은 결합 z로 비교합니다.
- **판정**: 기준을 넘고, 오더 2개 이상이면 오더 하나를 뺀 부모의 나머지 웨이퍼보다도 기준 z만큼 높아야 불량입니다. 더 구체적인 불량 조합으로 설명되면 '하위 기인'입니다.

In [ ]:
Z95, Z998, FWER_ALPHA = 1.959964, 3.090232, 0.05
SQRT2 = math.sqrt(2)
_A = (-39.69683028665376, 220.9460984245205, -275.9285104469687, 138.357751867269, -30.66479806614716, 2.506628277459239)
_B = (-54.47609879822406, 161.5858368580409, -155.6989798598866, 66.80131188771972, -13.28068155288572)
_C = (-0.007784894002430293, -0.3223964580411365, -2.400758277161838, -2.549732539343734, 4.374664141464968, 2.938163982698783)
_D = (0.007784695709041462, 0.3224671290700398, 2.445134137142996, 3.754408661907416)
_LG = (76.18009172947146, -86.50532032941677, 24.01409824083091, -1.231739572450155, 0.1208650973866179e-2, -0.5395239384953e-5)


def erfc(x):
    x = np.asarray(x, dtype=float)
    z = np.abs(x)
    t = 1 / (1 + 0.5 * z)
    poly = 1.00002368 + t * (0.37409196 + t * (0.09678418 + t * (-0.18628806 + t * (0.27886807 + t * (-1.13520398 + t * (1.48851587 + t * (-0.82215223 + t * 0.17087277)))))))
    r = t * np.exp(-z * z - 1.26551223 + t * poly)
    return np.where(x >= 0, r, 2 - r)


def norm_cdf(z):
    return 0.5 * erfc(-np.asarray(z, dtype=float) / SQRT2)


def norm_sf(z):
    return 0.5 * erfc(np.asarray(z, dtype=float) / SQRT2)


def norm_inv(p):
    if p <= 0:
        return -math.inf
    if p >= 1:
        return math.inf
    if p < 0.02425:
        q = math.sqrt(-2 * math.log(p))
        x = (((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    elif p <= 1 - 0.02425:
        q = p - 0.5
        r = q * q
        x = ((((((_A[0] * r + _A[1]) * r + _A[2]) * r + _A[3]) * r + _A[4]) * r + _A[5]) * q) / (((((_B[0] * r + _B[1]) * r + _B[2]) * r + _B[3]) * r + _B[4]) * r + 1)
    else:
        q = math.sqrt(-2 * math.log(1 - p))
        x = -(((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    e = float(norm_cdf(x)) - p
    u = e * math.sqrt(2 * math.pi) * math.exp((x * x) / 2)
    return x - u / (1 + (x * u) / 2)


def norm_isf(p):
    """상단 꼬리확률 → z (배열, 아주 작은 p도 정밀하게)"""
    p = np.clip(np.atleast_1d(np.asarray(p, dtype=float)), 1e-300, 1 - 1e-16)
    x = np.empty_like(p)
    lo, hi = p < 0.02425, p > 1 - 0.02425
    mid = ~(lo | hi)
    q = np.sqrt(-2 * np.log(p[lo]))
    x[lo] = (((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    q = p[mid] - 0.5
    r = q * q
    x[mid] = ((((((_A[0] * r + _A[1]) * r + _A[2]) * r + _A[3]) * r + _A[4]) * r + _A[5]) * q) / (((((_B[0] * r + _B[1]) * r + _B[2]) * r + _B[3]) * r + _B[4]) * r + 1)
    q = np.sqrt(-2 * np.log(1 - p[hi]))
    x[hi] = -(((((_C[0] * q + _C[1]) * q + _C[2]) * q + _C[3]) * q + _C[4]) * q + _C[5]) / ((((_D[0] * q + _D[1]) * q + _D[2]) * q + _D[3]) * q + 1)
    with np.errstate(all='ignore'):
        e = norm_cdf(x) - p
        u = e * math.sqrt(2 * math.pi) * np.exp((x * x) / 2)
        refined = x - u / (1 + (x * u) / 2)
    return -np.where(np.isfinite(refined) & (p > 1e-250), refined, x)


def lgamma(x):
    x = np.asarray(x, dtype=float)
    y = x.copy()
    tmp = x + 5.5 - (x + 0.5) * np.log(x + 5.5)
    ser = 1.000000000190015
    for g in _LG:
        y = y + 1
        ser = ser + g / y
    return -tmp + np.log((2.5066282746310005 * ser) / x)


def betacf(a, b, x):
    a, b, x = np.broadcast_arrays(*(np.asarray(v, dtype=float) for v in (a, b, x)))
    MAXIT, EPS, FPMIN = 300, 3e-14, 1e-300
    qab, qap, qam = a + b, a + 1, a - 1
    floor = lambda v: np.where(np.abs(v) < FPMIN, FPMIN, v)
    c = np.ones_like(x)
    d = 1 / floor(1 - (qab * x) / qap)
    h_ = d.copy()
    live = np.ones(x.shape, dtype=bool)
    with np.errstate(all='ignore'):
        for m in range(1, MAXIT + 1):
            m2 = 2 * m
            aa = (m * (b - m) * x) / ((qam + m2) * (a + m2))
            d1 = 1 / floor(1 + aa * d)
            c1 = floor(1 + aa / c)
            h1 = h_ * (d1 * c1)
            aa = (-(a + m) * (qab + m) * x) / ((a + m2) * (qap + m2))
            d1 = 1 / floor(1 + aa * d1)
            c1 = floor(1 + aa / c1)
            de = d1 * c1
            h1 = h1 * de
            c, d, h_ = np.where(live, c1, c), np.where(live, d1, d), np.where(live, h1, h_)
            live &= ~(np.abs(de - 1) < EPS)
            if not live.any():
                break
    return h_


def ibeta(x, a, b):
    x, a, b = np.broadcast_arrays(*(np.atleast_1d(np.asarray(v, dtype=float)) for v in (x, a, b)))
    out = np.where(x <= 0, 0.0, 1.0)
    mid = (x > 0) & (x < 1)
    if mid.any():
        xm, am, bm = x[mid], a[mid], b[mid]
        bt = np.exp(lgamma(am + bm) - lgamma(am) - lgamma(bm) + am * np.log(xm) + bm * np.log(1 - xm))
        front = xm < (am + 1) / (am + bm + 2)
        r = np.empty_like(xm)
        if front.any():
            r[front] = (bt[front] * betacf(am[front], bm[front], xm[front])) / am[front]
        if (~front).any():
            r[~front] = 1 - (bt[~front] * betacf(bm[~front], am[~front], 1 - xm[~front])) / bm[~front]
        out[mid] = r
    return out


def t_sf(t, df):
    t, df = np.atleast_1d(np.asarray(t, dtype=float)), np.atleast_1d(np.asarray(df, dtype=float))
    p = 0.5 * ibeta(df / (df + t * t), df / 2, 0.5)
    return np.where(t >= 0, p, 1 - p)


def f_sf(f, d1, d2):
    f = np.atleast_1d(np.asarray(f, dtype=float))
    with np.errstate(all='ignore'):
        p = ibeta(d2 / (d2 + d1 * f), d2 / 2, d1 / 2)
    return np.where(f <= 0, 1.0, p)


def var_of(n, s, ss):
    return max(0.0, (ss - (s * s) / n) / (n - 1)) if n > 1 else 0.0


def anova(n, s, ss):
    keep_ = n > 0
    n, s, ss = n[keep_].tolist(), s[keep_].tolist(), ss[keep_].tolist()
    k, N_ = len(n), sum(n)
    if k < 2 or N_ - k < 1:
        return None
    grand = sum(s) / N_
    ssb = sum(ni * (si / ni - grand) ** 2 for ni, si in zip(n, s))
    ssw = sum(ssi - ni * (si / ni) ** 2 for ni, si, ssi in zip(n, s, ss))
    if ssw <= 0:
        return None
    d1, d2 = k - 1, N_ - k
    F = ssb / d1 / (ssw / d2)
    return {'F': F, 'd1': d1, 'd2': d2, 'p': float(f_sf(F, d1, d2)[0])}


def welch(na, sa, ssa, nb, sb, ssb):
    """a가 b보다 큰지 단측 Welch t (요약통계 배열)"""
    na, sa, ssa, nb, sb, ssb = (np.atleast_1d(np.asarray(v, dtype=float)) for v in (na, sa, ssa, nb, sb, ssb))
    t, df, p = np.zeros_like(na), np.ones_like(na), np.ones_like(na)
    ok = (na >= 2) & (nb >= 2)
    with np.errstate(all='ignore'):
        va = np.maximum(0, (ssa - (sa * sa) / na) / (na - 1)) / na
        vb = np.maximum(0, (ssb - (sb * sb) / nb) / (nb - 1)) / nb
        se = np.sqrt(va + vb)
        ok &= se != 0
        t[ok] = (sa[ok] / na[ok] - sb[ok] / nb[ok]) / se[ok]
        df[ok] = (va[ok] + vb[ok]) ** 2 / ((va[ok] * va[ok]) / (na[ok] - 1) + (vb[ok] * vb[ok]) / (nb[ok] - 1))
    if ok.any():
        p[ok] = t_sf(t[ok], df[ok])
    return t, df, p


def quantile(v, p):
    if not len(v):
        return math.nan
    h_ = (len(v) - 1) * p
    lo = math.floor(h_)
    return float(v[lo] + (h_ - lo) * (v[min(lo + 1, len(v) - 1)] - v[lo]))

In [ ]:
def combo_key(s, items):
    return f'{s}|' + '|'.join(f'{q}:{u}' for q, u in items)


def binom_fix(n, x, p0):
    """bad(0/1) z의 꼬리 보정 배수. bad 비율이 낮고 웨이퍼가 적으면 bad 장수 분포가 오른쪽으로 길어서
    정규 근사 z가 실제보다 커진다. 정확한 이항 꼬리확률 P(bad ≥ x)로 구한 z ÷ 정규 근사 z를 곱해 줄인다(1보다 크게 하지는 않음)"""
    n, x = np.asarray(n, dtype=float), np.rint(np.asarray(x, dtype=float))
    z_norm = (x - n * p0) / np.sqrt(n * p0 * (1 - p0))
    fix = np.ones_like(n)
    up = z_norm > 0.5
    if up.any() and 0 < p0 < 1:
        nu, xu = n[up], x[up]
        ge = lambda k: np.where(k <= 0, 1.0, ibeta(np.full_like(nu, p0), np.maximum(k, 1), np.maximum(nu - k + 1, 1e-9)))
        fix[up] = np.clip(np.minimum(norm_isf(np.maximum(ge(xu), 1e-300)), 38.0) / z_norm[up], 0.0, 1.0)
    return fix


def lot_model(Y, lot_idx, L):
    """랏 랜덤 효과 모형: y = μ + 조합 효과 + 랏 효과 + 웨이퍼 오차.
    랏 간 분산 σa² · 랏 안 분산 σe²를 일원분산분석으로 구하고, 조합 z 계산에 쓰는 웨이퍼 · 랏별 값을 미리 만든다.
    θ = 1 − √(σe² ÷ (σe² + 랏 장수·σa²)): 랏 평균을 얼마나 빼고 비교할지 (랏 효과가 클수록 1에 가깝다)"""
    nl = np.bincount(lot_idx, minlength=L).astype(float)
    ybar = np.divide(np.bincount(lot_idx, weights=Y, minlength=L), nl, out=np.zeros(L), where=nl > 0)
    k_l, Nt = int((nl > 0).sum()), nl.sum()
    msb = float((nl * (ybar - Y.mean()) ** 2).sum()) / max(1, k_l - 1)
    msw = float(((Y - ybar[lot_idx]) ** 2).sum()) / max(1, Nt - k_l)
    n0 = (Nt - (nl ** 2).sum() / Nt) / max(1, k_l - 1)
    sa2, se2 = max(0.0, (msb - msw) / n0), msw
    theta = np.where(nl > 0, 1 - np.sqrt(se2 / (se2 + nl * sa2)), 0.0)
    f, w1 = theta * (2 - theta), nl * (1 - theta) ** 2
    return {'sa2': sa2, 'se2': se2, 'icc': sa2 / (sa2 + se2) if sa2 + se2 > 0 else 0.0,
            'gw': (ybar * f)[lot_idx], 'qw': ((1 - theta) ** 2)[lot_idx],
            'hl': np.divide(f, nl, out=np.zeros(L), where=nl > 0), 'mu': float((w1 * ybar).sum() / w1.sum())}


def analyze(data, min_n=20, max_depth=3, full_depth=2, lots=True):
    """조합 생성과 조합별 지표 (웹 js/analysis.js analyze와 같은 조합 · 관문 로직).
    신호(y_value, bad)마다 조합의 z를 랏 랜덤 효과 모형으로 구하고(GLS), 결합 z = (z_y + z_bad) ÷ √(2 + 2r)로 판정한다.
    - 한 랏의 웨이퍼가 유닛끼리 나뉘면 같은 랏 안에서 비교해 랏 차이가 상쇄되고, 랏이 통째로 한 유닛을 지나면 랏 단위 증거만 남는다.
    - r은 두 z가 우연히 같이 움직이는 정도(오더 1개 조합에서 잰 상관, 0~0.95)라 같은 정보를 두 번 세지 않게 한다.
    - 유효 웨이퍼 수 = y_value 비교가 가진 정보량(웨이퍼 장수 단위). good_bad가 없으면 z_y만 쓴다.
    lots=False면 웨이퍼끼리 독립으로 보는 z (웹과 같은 계산)"""
    N, steps, assign = data['N'], data['steps'], data['assign']
    lot_idx, L = data['lot_idx'], data['n_lots']
    sig = {'y': data['Y']} | ({'b': data['B']} if data.get('B') is not None else {})
    models = {k_: lot_model(v, lot_idx, L) if lots else None for k_, v in sig.items()}
    base = {k_: (float(v.mean()), float(v.std(ddof=1))) for k_, v in sig.items()}
    Y = sig['y']
    Y2 = Y * Y
    mu, sd = base['y']
    bad01 = np.nan_to_num(data['bad']) if 'b' in sig else None
    p0 = float(bad01.mean()) if 'b' in sig else 0.0

    def cell_keys(s, qs):
        A = assign[s][list(qs)]
        dims = [len(steps[s]['seqs'][q]['units']) for q in qs]
        ok = np.all(A != MISSING, axis=0)
        key = np.zeros(N, dtype=np.int64)
        for i, d in enumerate(dims):
            key = key * d + A[i]
        return dims, ok, key

    def cells(s, qs):
        """오더 묶음 qs에서 웨이퍼가 실제로 지난 칸(유닛 조합)만: 칸 번호(오름차순) · 칸마다 웨이퍼 수 · 웨이퍼 → 칸 순번.
        가능한 칸 수(유닛 수의 곱)가 아무리 커도 웨이퍼 수만큼만 계산한다"""
        dims, ok, key = cell_keys(s, qs)
        u, inv, cnt = np.unique(key[ok], return_inverse=True, return_counts=True)
        return dims, ok, u, inv, cnt

    def stats(ok, inv, m, idx):
        """칸마다 신호별 합(랏 모형이면 랏별 값의 합도)을 구해 idx 칸만 돌려준다"""
        w = lambda v: np.bincount(inv, weights=v[ok], minlength=m)[idx]
        out, per_lot = {'ss': w(Y2)}, None
        if 'b' in sig:
            out['x_bad'] = w(bad01)                   # 원래 bad 장수 (0/1, part 보정 전): 이항분포 꼬리 보정용
        for k_, v in sig.items():
            out[f'sum_{k_}'] = w(v)
            lm = models[k_]
            if lm is not None:
                if per_lot is None:                  # (칸, 랏)마다 웨이퍼 수² — 두 신호가 같이 쓴다
                    v_, c_ = np.unique(inv * L + lot_idx[ok], return_counts=True)
                    per_lot = (v_ // L, v_ % L, c_.astype(float) ** 2)
                cell, lot, c2 = per_lot
                out[f'G_{k_}'], out[f'Q_{k_}'] = w(lm['gw']), w(lm['qw'])
                out[f'M2h_{k_}'] = np.bincount(cell, weights=c2 * lm['hl'][lot], minlength=m)[idx]
        return out

    def sig_z(k_, nn, col):
        """조합들의 신호 k_ z와 정보량(유효 웨이퍼 수)"""
        lm, (m_, s_) = models[k_], base[k_]
        sm = col(f'sum_{k_}')
        if lm is None:
            return (sm / nn - m_) / (s_ / np.sqrt(nn)), nn
        n_eff = nn - col(f'M2h_{k_}')
        return (sm - col(f'G_{k_}') - col(f'Q_{k_}') * lm['mu']) / np.sqrt(lm['se2'] * n_eff), n_eff

    def decode(qs, dims, key):
        units = []
        for d in reversed(dims):
            units.append(key % d)
            key //= d
        return tuple((q, int(u)) for q, u in zip(qs, reversed(units)))

    def parent_keys(dims, keys, i):
        """칸 번호에서 i번째 오더 자리를 뺀 부모 칸 번호"""
        low = math.prod(dims[i + 1:])
        return (keys // (low * dims[i])) * low + keys % low

    seq_info = []
    for s, st in enumerate(steps):
        rows = []
        for q, sq in enumerate(st['seqs']):
            a = assign[s][q]
            ok, nu = a != MISSING, len(sq['units'])
            n_u = np.bincount(a[ok], minlength=nu)
            s_u, ss_u = np.bincount(a[ok], weights=Y[ok], minlength=nu), np.bincount(a[ok], weights=Y2[ok], minlength=nu)
            units = [{'name': name, 'n': int(n_u[u]), 'mean': s_u[u] / n_u[u] if n_u[u] else math.nan,
                      'sd': math.sqrt(var_of(int(n_u[u]), s_u[u], ss_u[u]))} for u, name in enumerate(sq['units'])]
            testable = nu >= 2
            rows.append({'name': sq['name'], 'units': units, 'testable': testable, 'anova': anova(n_u, s_u, ss_u) if testable else None})
        seq_info.append(rows)

    combos, by_key, r = [], {}, 0.0
    flagged = {}                        # (스텝, 오더 묶음) → z 1.96을 넘은 칸 번호들 (다음 깊이의 관문)
    has_big = set()                     # 웨이퍼 MIN_N장 이상인 칸이 있는 (스텝, 오더 묶음). 부모에 없으면 자식에도 없다
    multi = [[q for q, sq in enumerate(st['seqs']) if len(sq['units']) >= 2] for st in steps]
    search = [0] * (max_depth + 1)
    for k in range(1, max_depth + 1):
        gated = k > full_depth
        cand = []
        for s in range(len(steps)):
            if len(multi[s]) < k:
                continue
            for qs in combinations(multi[s], k):
                subs = [(s, qs[:i] + qs[i + 1:]) for i in range(k)]
                if k > 1 and not all(p in has_big for p in subs):
                    continue                    # 오더 하나를 뺀 묶음에 큰 칸이 없으면 이 묶음에도 없다
                dims, ok, u, inv, cnt = cells(s, qs)
                big = np.flatnonzero(cnt >= min_n)
                search[k] += len(big)           # 기준 z의 보정에는 계산을 건너뛴 칸도 센다
                if len(big):
                    has_big.add((s, qs))
                if not len(big) or (gated and not any(p in flagged for p in subs)):
                    continue
                if gated:                       # 오더 하나를 뺀 부모 조합 중 하나라도 z 1.96을 넘은 칸만 계산
                    keep = np.zeros(len(big), dtype=bool)
                    for i, p in enumerate(subs):
                        if p in flagged:
                            keep |= np.isin(parent_keys(dims, u[big], i), flagged[p])
                    big = big[keep]
                    if not len(big):
                        continue
                cand.append((s, qs, dims, u[big], cnt[big], stats(ok, inv, len(u), big)))
        if not cand:
            continue
        col = lambda f: np.concatenate([c[5][f] for c in cand]).astype(float)
        nn = np.concatenate([c[4] for c in cand]).astype(float)
        sm, ss = col('sum_y'), col('ss')
        zy, n_eff = sig_z('y', nn, col)
        zb = sig_z('b', nn, col)[0] * binom_fix(nn, col('x_bad'), p0) if 'b' in sig else None
        if zb is not None and k == 1 and len(zy) > 2:   # 두 z의 상관: 오더 1개 조합(대부분 우연)에서 잰다
            r = float(np.clip(np.corrcoef(zy, zb)[0, 1], 0.0, 0.95))
        z = zy if zb is None else (zy + zb) / math.sqrt(2 + 2 * r)
        bad_rate = col('x_bad') / nn if 'b' in sig else None   # 원래(0/1) bad 비율 — 랭킹의 초과 bad 웨이퍼 계산용
        j = 0
        for s, qs, dims, keys, _, _ in cand:
            hot = []
            for key in keys.tolist():
                items = decode(qs, dims, key)
                c = {'id': len(combos), 'key': combo_key(s, items), 'step': s, 'k': k, 'items': items,
                     'n': int(nn[j]), 'sum': float(sm[j]), 'ss': float(ss[j]), 'mean': float(sm[j] / nn[j]),
                     'z': float(z[j]), 'z_y': float(zy[j]), 'z_b': float(zb[j]) if zb is not None else math.nan,
                     'bad_rate': float(bad_rate[j]) if bad_rate is not None else math.nan,
                     'n_eff': float(n_eff[j])}
                combos.append(c)
                by_key[c['key']] = c
                if c['z'] > Z95:
                    hot.append(key)
                j += 1
            if hot:
                flagged[(s, qs)] = np.array(hot, dtype=np.int64)
    depth_count = [0] * (max_depth + 1)
    for c in combos:
        depth_count[c['k']] += 1
    return {'mu': mu, 'sd': sd, 'N': N, 'max_depth': max_depth, 'models': models, 'r': r, 'p0': p0 if 'b' in sig else None,
            'rho': models['y']['icc'] if lots else 0.0, 'rho_b': models['b']['icc'] if lots and 'b' in sig else 0.0,
            'band_sd': math.sqrt(models['y']['se2']) if lots else sd,
            'seq_info': seq_info, 'combos': combos, 'by_key': by_key, 'depth_count': depth_count, 'search_space': search}


def thresholds(res):
    """오더 수별 기준 z: 가능한 조합 수만큼 보정(Bonferroni, 전체에서 하나라도 우연히 넘을 확률 5%)"""
    return [0.0] + [norm_inv(1 - FWER_ALPHA / m) if m else Z998 for m in res['search_space'][1:]]


def members_of(d, c):
    """조합의 웨이퍼 (불리언 배열)"""
    A = d['assign'][c['step']]
    m = np.ones(d['N'], dtype=bool)
    for q, u in c['items']:
        m &= A[q] == u
    return m


def lot_counts(d, m):
    """웨이퍼 묶음의 랏별 장수"""
    return np.bincount(d['lot_idx'][m], minlength=d['n_lots'])


def mean_var(r, v, n, s2):
    """평균의 분산 v(웨이퍼끼리 독립일 때)에 랏 상관(설계 효과 = 1 + ICC × (Σ 랏별 장수² ÷ 장수 − 1))을 반영"""
    return v * (1 + r['rho'] * (s2 / n - 1) if r['rho'] else 1.0)


def sig_values(d, k_):
    """신호 값: 'y' = 분석용 y_value, 'b' = bad(0/1, part 보정했으면 보정값)"""
    return d['Y'] if k_ == 'y' else d['B']


def sig_mean_z(d, r, k_, m):
    """웨이퍼 묶음 m에서 신호 k_의 평균이 전체보다 높은지의 z (조합의 z와 같은 계산)"""
    v, lm, n = sig_values(d, k_), r['models'][k_], int(m.sum())
    if lm is None:
        return float((v[m].mean() - v.mean()) / (v.std(ddof=1) / math.sqrt(n)))
    n_eff = n - float((lot_counts(d, m) ** 2 * lm['hl']).sum())
    return float((v[m].sum() - lm['gw'][m].sum() - lm['qw'][m].sum() * lm['mu']) / math.sqrt(lm['se2'] * n_eff))


def sig_diff_z(d, r, k_, a, b):
    """두 웨이퍼 묶음에서 신호 k_의 평균 차이(a − b)의 z. 랏 모형이면 두 묶음 웨이퍼만 놓고 랏 랜덤 효과 GLS로 비교해서,
    같은 랏을 나눠 가진 만큼은 랏 차이가 상쇄되고 랏이 갈리는 만큼은 랏 단위 증거만 남는다. 아니면 Welch t"""
    v, lm = sig_values(d, k_), r['models'][k_]
    if lm is None:
        var = v[a].var(ddof=1) / a.sum() + v[b].var(ddof=1) / b.sum()
        return float((v[a].mean() - v[b].mean()) / math.sqrt(var)) if var > 0 else 0.0
    u = a | b
    lot, y, xa, L = d['lot_idx'][u], v[u], a[u], d['n_lots']
    nl = np.bincount(lot, minlength=L).astype(float)
    ybar = np.divide(np.bincount(lot, weights=y, minlength=L), nl, out=np.zeros(L), where=nl > 0)
    th = np.where(nl > 0, 1 - np.sqrt(lm['se2'] / (lm['se2'] + nl * lm['sa2'])), 0.0)
    f, w1 = th * (2 - th), nl * (1 - th) ** 2
    m = np.bincount(lot[xa], minlength=L).astype(float)
    A = float(y[xa].sum() - (m * ybar * f).sum())
    B = float(m.sum() - (m * m * np.divide(f, nl, out=np.zeros(L), where=nl > 0)).sum())
    X, Cc, Cy = float((m * (1 - th) ** 2).sum()), float(w1.sum()), float((w1 * ybar).sum())
    den = B - X * X / Cc
    return float((A - X * Cy / Cc) / den / math.sqrt(lm['se2'] / den)) if den > 0 else 0.0


def combine(r, zs):
    """신호별 z를 하나로: (z_y + z_bad) ÷ √(2 + 2r). good_bad가 없으면 z_y 그대로"""
    return zs[0] if len(zs) == 1 else (zs[0] + zs[1]) / math.sqrt(2 + 2 * r['r'])


def mean_z(d, r, m):
    """웨이퍼 묶음이 전체보다 나쁜지의 결합 z (조합의 z와 같은 계산)"""
    if m.sum() < 2:
        return 0.0
    zs = [sig_mean_z(d, r, k_, m) for k_ in r['models']]
    if len(zs) > 1:
        bad01 = np.nan_to_num(d['bad'])
        zs[1] *= float(binom_fix([m.sum()], [bad01[m].sum()], bad01.mean())[0])
    return combine(r, zs)


def diff_z(d, r, a, b):
    """두 웨이퍼 묶음의 차이(a가 b보다 나쁜지)의 결합 z"""
    return combine(r, [sig_diff_z(d, r, k_, a, b) for k_ in r['models']]) if a.sum() >= 2 and b.sum() >= 2 else 0.0


def classify(res, zk, d):
    """판정: 정상 · 경고 · 불량 · 상속 · 하위 기인 (웹 js/analysis.js classify와 같은 규칙).
    부모와의 비교(리프트)와 하위 기인은 기준을 넘은 조합에만 필요해서, 그 조합들만 웨이퍼로 직접 계산한다"""
    combos, by_key = res['combos'], res['by_key']
    mem = {}

    def member(c):
        if c['key'] not in mem:
            mem[c['key']] = members_of(d, c)
        return mem[c['key']]

    for c in combos:
        c['over'] = c['z'] > zk[c['k']]
        c['explained_by'], c['children'] = None, []
        c['status'] = ('bad' if c['k'] == 1 else None) if c['over'] else ('warn' if c['z'] > Z95 else 'normal')
    for c in combos:
        if not c['over'] or c['k'] == 1:
            continue
        m = member(c)
        c['parents'] = []
        for i in range(c['k']):
            p_items = c['items'][:i] + c['items'][i + 1:]
            pm = members_of(d, {'step': c['step'], 'items': p_items})
            rest = pm & ~m
            c['parents'].append({'key': combo_key(c['step'], p_items), 'items': p_items, 'n': int(pm.sum()), 'mean': float(d['Y'][pm].mean()),
                                 'rest_n': int(rest.sum()), 'rest_mean': float(d['Y'][rest].mean()) if rest.any() else math.nan,
                                 't': diff_z(d, res, m, rest)})
            if (P := by_key.get(c['parents'][-1]['key'])) is not None:
                P['children'].append(c['id'])
        c['lift_t'] = min(p['t'] for p in c['parents'])
        c['status'] = 'bad' if c['lift_t'] > zk[c['k']] else 'inherited'
    for k in range(res['max_depth'] - 1, 0, -1):
        for c in combos:
            if c['k'] != k or c['status'] != 'bad':
                continue
            for cid in c['children']:
                ch = combos[cid]
                if ch['status'] not in ('bad', 'explained'):
                    continue
                rest = member(c) & ~member(ch)
                if rest.sum() >= 2 and mean_z(d, res, rest) <= zk[k]:
                    c['status'], c['explained_by'] = 'explained', ch['id']
                    break
    count = lambda st: sum(c['status'] == st for c in combos)
    return {'over': sum(c['over'] for c in combos), 'bad': count('bad'), 'inherited': count('inherited'), 'explained': count('explained')}


def build_index(r):
    """기준 밖 조합을 (스텝, 오더, 유닛)으로 바로 찾는 색인. 후보 찾기가 전체 조합을 매번 훑지 않게 한다"""
    idx = defaultdict(list)
    for b in r['combos']:
        if b['over']:
            for it in b['items']:
                idx[(b['step'], it)].append(b)
    r['over_index'] = idx

In [ ]:
RANK_NAME = '초과 bad 웨이퍼' if data['B'] is not None else '영향 (웨이퍼 × 차이)'   # 랭킹 · 대표 선택 기준
STATUS_LABEL = {'bad': '불량', 'inherited': '관련 조합 · 상속', 'explained': '관련 조합 · 하위 기인', 'warn': '정상 범위', 'normal': '정상 범위'}
STATUS_SHORT = {'bad': '불량', 'inherited': '상속', 'explained': '하위 기인', 'warn': '기준 아래', 'normal': '기준 아래'}
STEP_NAME = None if STEP is None else re.sub(STEP_SUFFIX, '', str(STEP)) if STEP_SUFFIX else str(STEP)   # 'ex100000_3'을 줘도 스텝 ex100000
in_scope = lambda c: STEP_NAME is None or steps[c['step']]['name'] == STEP_NAME


def path_label(st, items, sep=' → '):
    return sep.join(f"{st['seqs'][q]['name']}:{st['seqs'][q]['units'][u]}" for q, u in items)


def metric(c, r):
    """랭킹 · 대표 선택 기준: 초과 bad 웨이퍼 = 웨이퍼 × (bad 비율 − 전체 bad 비율), 즉 고치면 줄어드는 bad 웨이퍼 수.
    good_bad가 없으면 웨이퍼 × y_value 차이"""
    return c['n'] * (c['bad_rate'] - r['p0']) if r['p0'] is not None else c['n'] * (c['mean'] - r['mu'])


def candidates(r, c):
    """고른 조합과 같은 스텝에서 유닛을 하나 이상 공유하고 기준을 넘은 조합 (색인으로 찾음)"""
    found = {b['id']: b for it in c['items'] for b in r['over_index'].get((c['step'], it), ())}
    return [found[i] for i in sorted(found)]


def representative(r, zk_, c):
    """기준 밖 점의 대표 불량 대상: 같은 스텝에서 유닛을 공유하는 불량 조합 중 초과 bad 웨이퍼가 가장 많은 것"""
    bad = [b for b in candidates(r, c) if b['status'] == 'bad']
    return max(bad, key=lambda b: metric(b, r)) if bad else None


def reason(r, zk_, st, b, target):
    if b['status'] == 'bad':
        return '대표 불량 대상' if b is target else '불량 (다른 대상)'
    if b['status'] == 'explained':
        return f"하위 기인: {path_label(st, r['combos'][b['explained_by']]['items'])} 때문"
    pr = min(b['parents'], key=lambda p: p['t'])
    return f"상속: {path_label(st, pr['items'])} 대비 t {pr['t']:.1f} ≤ 기준 {zk_[b['k']]:.1f}"


def combo_row(d, r, zk_, c):
    st = d['steps'][c['step']]
    m = members_of(d, c)
    desc = ' / '.join(dict.fromkeys(st['seqs'][q]['desc'] for q, _ in c['items']))   # 경로가 지난 오더(step_seq)의 step_desc
    row = {'스텝': st['name'], 'step_desc': desc, '경로': path_label(st, c['items']), '오더 수': c['k'], '웨이퍼': c['n']}
    if r['models']['y'] is not None:                         # 랏 차이를 뺀 비교가 가진 정보량(웨이퍼 장수 단위)
        row['유효 웨이퍼'] = round(c['n_eff'])
    row |= {'랏 수': len(np.unique(d['lots'][m])), f'평균 {Y_NAME}': c['mean']}
    if Y_NAME != 'y_value':                                  # 값을 바꿔 분석했으면 원래 y_value 평균도 함께
        row['평균 y_value'] = float(np.nanmean(d['value'][m]))
    if d['B'] is not None:
        row['bad 비율'] = float(np.nanmean(d['bad'][m]))
    row[RANK_NAME] = round(metric(c, r), 1)
    row |= {'차이': c['mean'] - r['mu'], 'z ÷ 기준 z': c['z'] / zk_[c['k']], 'z y_value': c['z_y']}
    if d['B'] is not None:
        row['z bad'] = c['z_b']
    return row


def cross_step_check(d, r, zk_):
    """스텝 간 규칙: 다른 스텝의 불량 조합 A와 웨이퍼가 겹치는 불량 조합 B에 대해, A를 지나지 않은 웨이퍼만 놓고
    B가 나머지보다 높은지(Welch t, 조합의 z와 같은 보정) 본다. B의 효과가 사라지고(t ≤ 1.96) A는 B 없이도 기준을 넘으면 B를 'A 기인'으로 본다"""
    bads = [c for c in r['combos'] if c['status'] == 'bad']
    mem = {c['key']: members_of(d, c) for c in bads}

    def t_without(c, other):
        return diff_z(d, r, mem[c['key']] & ~mem[other['key']], ~mem[c['key']] & ~mem[other['key']])

    out = {}
    if not bads:
        return out
    M = np.array([mem[c['key']] for c in bads], dtype=np.float32)
    overlap = (M @ M.T) > 0                       # 웨이퍼가 겹치는 쌍을 한 번에
    for i, b in enumerate(bads):
        cause = [(t_without(b, a), a) for j, a in enumerate(bads) if overlap[i, j] and a['step'] != b['step']]
        cause = [(t, a) for t, a in cause if t <= Z95 and t_without(a, b) > zk_[a['k']]]
        if cause:
            out[b['key']] = min(cause, key=lambda ta: ta[0])[1]
    return out

In [ ]:
CHART_SEP = ' ⟶ '       # 차트 안 경로 구분자. 나눔고딕의 '→'는 아주 작게 찍혀서 차트에서는 긴 화살표를 쓴다 (표는 '→')


def mix(c1, c2, t):
    a, b = np.array(to_rgb(c1)), np.array(to_rgb(c2))
    return tuple(a * t + b * (1 - t))


def fmt_p(p):
    if p is None or not math.isfinite(p):
        return '–'
    return f'{p:.0e}' if p < 1e-4 else f'{p:.4f}'


def plain_log(axis):
    axis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
    axis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}' if v >= 1 else f'{v:g}'))
    axis.set_minor_locator(NullLocator())


def label_points(ax, pts, gap=14):
    """불량 점 오른쪽에 이름. 점이 몰려 있으면 라벨을 무리의 오른쪽 바깥에 쌓고 가는 선으로 잇는다"""
    if not pts:
        return
    ax.autoscale_view()
    disp = ax.transData.transform([(x, y) for x, y, _ in pts])
    placed = []
    for i in np.argsort(disp[:, 1]):
        x0, y0 = disp[i]
        near = [j for j in range(len(pts)) if abs(disp[j][0] - x0) < 70 and abs(disp[j][1] - y0) < 20]
        lx, ly = max(disp[j][0] for j in near) + 9, y0
        for px, py in sorted(placed, key=lambda p: p[1]):
            if abs(lx - px) < 70 and abs(ly - py) < gap:
                ly = py + gap
        placed.append((lx, ly))
        x, y, text = pts[i]
        moved = len(near) > 1 or ly != y0
        ax.annotate(text, (x, y), xytext=(lx - x0 + (6 if moved else 0), ly - y0), textcoords='offset pixels', va='center',
                    fontsize=9, fontweight='bold', color=C['ink'], zorder=6,
                    arrowprops={'arrowstyle': '-', 'color': C['ink2'], 'lw': 0.8, 'shrinkA': 1, 'shrinkB': 5} if moved else None)


def plot_unified(ax, d, r, zk_, cross=None, skip_label=(), title=''):
    """모든 스텝 · 모든 조합 크기를 한 장의 funnel로. 가로축은 유효 웨이퍼 수(랏 상관을 반영한 웨이퍼 수),
    띠는 우연 범위 μ ± z₁·σ/√n이다(σ는 랏 모형이면 랏 안 표준편차). 점의 높이를 z ÷ z_k만큼 띠에 맞춰 그려서(오더 1개는 실제 평균)
    띠 하나로 모든 크기를 판정한다(띠 위 ⇔ z > z_k). skip_label의 조합(다른 대상과 웨이퍼가 같아 한 줄로 묶인 것)에는 이름을 달지 않는다"""
    cross = cross or {}
    cs = [c for c in r['combos'] if in_scope(c)]
    mu_, z1 = r['mu'], zk_[1]
    half = lambda n: z1 * r['band_sd'] / np.sqrt(n)

    def xy(group):
        if not group:
            return np.array([]), np.array([])
        ne = np.array([c['n_eff'] for c in group])
        return ne, mu_ + np.array([c['z'] / zk_[c['k']] for c in group]) * half(ne)

    pt = lambda c: (c['n_eff'], float(xy([c])[1][0]))
    ns = [c['n_eff'] for c in cs]
    lo, hi = max(2, min(ns) * 0.9), max(ns) * 1.1
    grid = lo * (hi / lo) ** (np.arange(80) / 79)
    ax.fill_between(grid, mu_ - half(grid), mu_ + half(grid), color=C['band'], lw=0, zorder=0)
    for sign in (1, -1):
        ax.plot(grid, mu_ + sign * half(grid), color=C['axis'], lw=1, zorder=1)
    ax.plot([lo, hi], [mu_, mu_], color=C['axis'], lw=1, ls=(0, (4, 3)), zorder=1)
    ax.scatter(*xy([c for c in cs if not c['over']]), s=4, color=C['muted'], alpha=0.22, lw=0, zorder=1)
    over = [c for c in cs if c['over']]
    ax.scatter(*xy([c for c in over if c['status'] != 'bad']), s=46, facecolors='none', edgecolors=C['bad'], lw=1.4, zorder=3)
    ax.scatter(*xy([c for c in over if c['status'] == 'bad' and c['key'] not in cross]), s=80, color=C['bad'], edgecolors=C['surface'], lw=1.8, zorder=4)
    xc = [c for c in cs if c['key'] in cross]
    ax.scatter(*xy(xc), s=110, facecolors=C['surface'], edgecolors=C['bad'], lw=2.4, zorder=5)
    if LOGX:
        ax.set_xscale('log')
        plain_log(ax.xaxis)
    steps_ = d['steps']
    named = [(*pt(c), steps_[c['step']]['name']) for c in cs if c['status'] == 'bad' and c['key'] not in cross and c['key'] not in skip_label]
    if len(named) <= 12:
        label_points(ax, named)
    for c in xc:
        a = cross[c['key']]
        ax.annotate(f"{steps_[c['step']]['name']} · 다른 스텝 기인\n({steps_[a['step']]['name']} {path_label(steps_[a['step']], a['items'], CHART_SEP)})",
                    pt(c), xytext=(12, -2), textcoords='offset points', va='center', fontsize=9, color=C['ink'])
    handles = [Line2D([], [], marker='o', ls='', color=C['bad'], mec=C['surface'], ms=8, label='불량'),
               Line2D([], [], marker='o', ls='', mfc='none', mec=C['bad'], ms=7, label='관련 조합'),
               Line2D([], [], marker='o', ls='', color=C['muted'], ms=4, label='우연 범위 안'),
               Patch(color=C['band'], label='우연 범위')]
    if xc:
        handles.append(Line2D([], [], marker='o', ls='', mfc=C['surface'], mec=C['bad'], mew=2.2, ms=9, label='다른 스텝 기인'))
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.01, 1), borderaxespad=0)
    ax.set(title=title, xlabel='유효 웨이퍼 수 (랏 차이를 뺀 비교의 정보량)' if r['models']['y'] is not None else '조합의 웨이퍼 수',
           ylabel=f'결합 z를 평균 {Y_NAME} 눈금으로 환산' if len(r['models']) > 1 else f'평균 {Y_NAME} (기준 환산)')
    ax.grid(axis='x', visible=False)


def plot_flow(c):
    """스텝 흐름: 오더 × 유닛 박스를 유닛 평균으로 색칠하고 경로를 잇는다"""
    st, info = steps[c['step']], res['seq_info'][c['step']]
    hit = dict(c['items'])
    qs_all = list(range(len(st['seqs'])))
    shown = qs_all
    if len(qs_all) > FLOW_MAX_ORDERS:                   # 오더가 많으면 경로 → 유닛 간 차이가 있는 오더 → 나머지 순으로 고른다
        sig = [q for q in qs_all if q not in hit and info[q]['anova'] and info[q]['anova']['p'] < 0.05]
        rest = sorted((q for q in qs_all if q not in hit and q not in sig), key=lambda q: (len(st['seqs'][q]['units']) < 2, q))
        shown = sorted((sorted(hit) + sig + rest)[:max(FLOW_MAX_ORDERS, len(hit))])
    pos = {q: i for i, q in enumerate(shown)}
    maxlen = max(len(u) for q in shown for u in st['seqs'][q]['units'])
    colW, gap, boxH, boxGap, top, padX = max(84, 7 * maxlen + 18), 26, 22, 4, 42, 2
    nq, maxU = len(shown), max(len(st['seqs'][q]['units']) for q in shown)
    W = padX * 2 + nq * colW + (nq - 1) * gap
    H = top + maxU * (boxH + boxGap) + 4
    max_dev = max([abs(res['sd']) * 0.1] + [u['mean'] - res['mu'] for sq in info for u in sq['units'] if not math.isnan(u['mean'])])
    col_x = lambda q: padX + pos[q] * (colW + gap)
    box_y = lambda u: top + u * (boxH + boxGap)
    fig = plt.figure(figsize=(W / 72, (H + 26) / 72))
    ax = fig.add_axes((0, 0, 1, H / (H + 26)))
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.axis('off')
    note = f' · 오더 {len(qs_all)}개 중 {len(shown)}개 (경로와 유닛 간 차이가 있는 오더 우선)' if len(shown) < len(qs_all) else ''
    fig.text(0, 1, f"스텝 흐름 · {st['name']} · 유닛 색 = 평균{note}", ha='left', va='top', fontsize=11, fontweight='bold', color=C['ink'])
    for q in shown:
        sq = st['seqs'][q]
        on, cx = q in hit, col_x(q) + colW / 2
        an_ = info[q]['anova']
        sub = '유닛 1개' if not info[q]['testable'] else (f"p {fmt_p(an_['p'])}" if an_ and an_['p'] < 0.05 else '차이 없음')
        ax.text(cx, 15, f"오더 {sq['name'][1:]}", ha='center', va='baseline', fontsize=12, fontweight='bold', color=C['bad'] if on else C['ink2'])
        ax.text(cx, 30, sub, ha='center', va='baseline', fontsize=10, color=C['muted'])
        for ui, u in enumerate(info[q]['units']):
            sel = on and hit[q] == ui
            t = 0.0 if math.isnan(u['mean']) else min(1.0, max(0.0, (u['mean'] - res['mu']) / max_dev))
            fill = C['bad'] if sel else mix(C['bad'], C['mid'], round(t * 55) / 100)
            alpha = 1 if on else 0.45
            ax.add_patch(FancyBboxPatch((col_x(q), box_y(ui)), colW, boxH, boxstyle='round,pad=0,rounding_size=6',
                                        fc=fill, ec='none', alpha=alpha, zorder=4 if sel else 3))
            ax.text(cx, box_y(ui) + boxH / 2 + 4, u['name'], ha='center', va='baseline', fontsize=10,
                    fontweight='bold' if sel else 'normal', color='white' if sel else C['ink'], alpha=alpha, zorder=5)
    pts = sorted((col_x(q) + colW / 2, box_y(u) + boxH / 2) for q, u in c['items'])
    if len(pts) > 1:
        verts, codes = [pts[0]], [MplPath.MOVETO]
        for (x0, y0), (x1, y1) in zip(pts, pts[1:]):
            dx = (x1 - x0) / 2
            verts += [(x0 + dx, y0), (x1 - dx, y1), (x1, y1)]
            codes += [MplPath.CURVE4] * 3
        ax.add_patch(PathPatch(MplPath(verts, codes), fc='none', ec=C['bad'], lw=3, alpha=0.85, capstyle='round', zorder=2))
    plt.show()


def membership(c):
    """이 경로 / 오더 하나만 다른 웨이퍼 / 이 경로가 아닌 나머지 (조합의 오더를 모두 지난 웨이퍼 기준)"""
    A = data['assign'][c['step']][[q for q, _ in c['items']]]
    U = np.array([u for _, u in c['items']])[:, None]
    present = np.all(A != MISSING, axis=0)
    mism = A != U
    miss = mism.sum(axis=0)
    me = np.flatnonzero(present & (miss == 0))
    others = np.flatnonzero(present & (miss > 0))
    parents = [np.flatnonzero(present & (miss == 1) & mism[i]) for i in range(c['k'])] if c['k'] >= 2 else []
    return me, others, parents


def box_stats(idx):
    v = np.sort(Y[idx])
    q1, q3 = quantile(v, 0.25), quantile(v, 0.75)
    iqr = q3 - q1
    return {'n': len(v), 'mean': float(v.mean()), 'q1': q1, 'med': quantile(v, 0.5), 'q3': q3,
            'lo': max(float(v[0]), q1 - 1.5 * iqr), 'hi': min(float(v[-1]), q3 + 1.5 * iqr), 'p01': quantile(v, 0.005), 'p99': quantile(v, 0.995)}


def plot_dist(ax, c, me, others, parents):
    st = steps[c['step']]
    groups = [('이 경로', me, C['bad'])]
    if c['k'] >= 2:
        for i, (q, u) in enumerate(c['items']):
            same = path_label(st, c['items'][:i] + c['items'][i + 1:], sep=' · ')
            groups.append((f"{same} 같음 · {st['seqs'][q]['name']} 다름", parents[i], C['ink2']))
    groups.append(('이 경로가 아닌 웨이퍼', others, C['muted']))
    groups = [g for g in groups if len(g[1])]
    sums = [box_stats(idx) for _, idx, _ in groups]
    rng, h_ = np.random.default_rng(SEED), 0.46
    for gi, ((label, idx, col), s_) in enumerate(zip(groups, sums)):
        pick_ = idx[::max(1, math.ceil(len(idx) / 500))]
        ax.scatter(Y[pick_], gi + (rng.random(len(pick_)) - 0.5) * 0.5, s=5, color=col, alpha=0.45 if gi == 0 else 0.2, lw=0, zorder=1)
        ax.plot([s_['lo'], s_['hi']], [gi, gi], color=col, lw=1.2, zorder=2)
        ax.add_patch(Rectangle((s_['q1'], gi - h_ / 2), s_['q3'] - s_['q1'], h_, fc=col, alpha=0.16, ec='none', zorder=2))
        ax.add_patch(Rectangle((s_['q1'], gi - h_ / 2), s_['q3'] - s_['q1'], h_, fc='none', ec=col, lw=1.4, zorder=3))
        ax.plot([s_['med']] * 2, [gi - h_ / 2, gi + h_ / 2], color=col, lw=2.5, zorder=3)
        ax.scatter([s_['mean']], [gi], marker='D', s=48, color=col, edgecolors=C['surface'], lw=1.5, zorder=4)
        ax.annotate(f"{s_['mean']:.3g} · {s_['n']:,}장", (s_['mean'], gi - h_ / 2), xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9, fontweight='bold', color=C['ink'], zorder=5)
    ax.axvline(res['mu'], color=C['muted'], ls=(0, (4, 3)), lw=1, zorder=0)
    ax.set_yticks(range(len(groups)), [g[0] for g in groups])
    ax.set_ylim(len(groups) - 0.5, -0.6)
    lo, hi = min(s_['p01'] for s_ in sums), max(s_['p99'] for s_ in sums)
    ax.set_xlim(lo - (hi - lo) * 0.03, hi + (hi - lo) * 0.03)
    ax.set_xlabel(Y_NAME)
    ax.set_title('웨이퍼 분포 · 상자 = 사분위 · ◆ = 평균 · 점선 = 전체 평균')
    ax.grid(axis='y', visible=False)


def wafer_stat(m):
    """웨이퍼 묶음의 장수 · 평균 · 95% 신뢰구간 반폭 (랏 상관을 반영)"""
    n = int(m.sum())
    if n < 2:
        return n, math.nan, math.nan
    v = Y[m]
    s2 = float((lot_counts(data, m) ** 2).sum()) if res['rho'] else n
    return n, float(v.mean()), 1.96 * math.sqrt(mean_var(res, float(v.var(ddof=1)) / n, n, s2))


G1 = '① 대표 불량 대상 ●'
G2 = '② 관련 조합 ● ⟶ 대표 대상의 웨이퍼를 빼면 ○'
G3 = '③ 대표 대상을 더 좁힌 조합 ● · 대표 대상의 나머지 웨이퍼 ○'


def why_rows(T, cap=6):
    st = steps[T['step']]
    mT, Ti = members_of(data, T), set(T['items'])
    lab = lambda items: path_label(st, items, CHART_SEP)
    if T['k'] == 1:
        (q, u), = T['items']
        A = data['assign'][T['step']][q]
        rows = [(G1, f"{lab(T['items'])} · ○ 같은 오더의 다른 유닛", wafer_stat(mT), wafer_stat((A != MISSING) & (A != u)))]
    else:
        rows = [(G1, lab(T['items']), wafer_stat(mT), None)]
    rel = [b for b in candidates(res, T) if b['key'] != T['key']]
    wider, seen = [], set()
    for i in range(T['k'] if T['k'] >= 2 else 0):
        items = T['items'][:i] + T['items'][i + 1:]
        key = combo_key(T['step'], items)
        wider.append(res['by_key'].get(key) or {'step': T['step'], 'items': items, 'status': 'normal', 'key': key})
        seen.add(key)
    wider += sorted((b for b in rel if b['key'] not in seen and not Ti < set(b['items'])), key=lambda b: -b['z'])[:cap]
    for c in wider:
        m = members_of(data, c)
        rows.append((G2, f"{lab(c['items'])} · {STATUS_SHORT[c['status']]}", wafer_stat(m), wafer_stat(m & ~mT)))
    for c in sorted((b for b in rel if Ti < set(b['items'])), key=lambda b: -b['z'])[:cap]:
        m = members_of(data, c)
        rows.append((G3, f"{lab(c['items'])} · {STATUS_SHORT[c['status']]}", wafer_stat(m), wafer_stat(mT & ~m)))
    return rows


def plot_why(T):
    rows = why_rows(T)
    layout = []
    for g in dict.fromkeys(r[0] for r in rows):
        layout.append(('header', g))
        layout += [('row', r) for r in rows if r[0] == g]
    fig, ax = plt.subplots(figsize=(12, 1.2 + 0.38 * len(layout)))
    for yy, (kind, item) in enumerate(layout):
        if kind == 'header':
            continue
        g, _, a, b = item
        col = C['bad'] if g == G1 else C['ink2']
        counts = f'● {a[0]:,}장'
        if b is not None and not math.isnan(b[1]):
            ax.plot([a[1], b[1]], [yy, yy], color=C['axis'], lw=2, zorder=1)
            ax.plot([b[1] - b[2], b[1] + b[2]], [yy, yy], color=C['muted'], lw=7, alpha=0.22, solid_capstyle='round', zorder=1)
            ax.scatter([b[1]], [yy], s=70, facecolors=C['surface'], edgecolors=C['ink2'], lw=1.6, zorder=3)
            counts += f' · ○ {b[0]:,}장'
        ax.plot([a[1] - a[2], a[1] + a[2]], [yy, yy], color=col, lw=7, alpha=0.2, solid_capstyle='round', zorder=1)
        ax.scatter([a[1]], [yy], s=80, color=col, edgecolors=C['surface'], lw=1.5, zorder=4)
        ax.annotate(counts, (1, yy), xycoords=('axes fraction', 'data'), xytext=(8, 0), textcoords='offset points',
                    va='center', fontsize=8, color=C['muted'], annotation_clip=False)
    ax.axvline(res['mu'], color=C['muted'], lw=1, ls=(0, (4, 3)), zorder=0)
    ax.axvline(T['mean'], color=C['bad'], lw=1, ls=(0, (1, 2)), zorder=0)
    ax.set_yticks(range(len(layout)), [item if kind == 'header' else item[1] for kind, item in layout])
    for lab, (kind, _) in zip(ax.get_yticklabels(), layout):
        if kind == 'header':
            lab.set_fontweight('bold')
            lab.set_color(C['ink'])
    ax.set_ylim(len(layout) - 0.4, -0.9)
    for x, text, colr in ((res['mu'], '전체 평균', C['ink2']), (T['mean'], '대표 대상 평균', C['bad'])):
        ax.annotate(text, (x, -0.9), xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9, color=colr, annotation_clip=False)
    ax.set_xlabel(f'평균 {Y_NAME} · 띠 = 95% 신뢰구간 · 오른쪽 숫자 = 웨이퍼 수')
    ax.set_title(f"왜 대표 불량 대상인가 · {steps[T['step']]['name']} {path_label(steps[T['step']], T['items'], CHART_SEP)}", pad=22)
    ax.grid(axis='y', visible=False)
    plt.tight_layout()
    plt.show()


def plot_time(T):
    """시간 추이: 이 스텝의 track-in 시각 순서로 이 경로와 스텝 전체의 이동평균"""
    s = T['step']
    t = data['trackin'][s]
    present = np.any(data['assign'][s] != MISSING, axis=0) & ~np.isnat(t)
    me = members_of(data, T) & ~np.isnat(t)
    if me.sum() < 3:
        print('track-in 시각이 있는 웨이퍼가 적어 시간 추이를 건너뜁니다.')
        return
    all_s = pd.Series(Y[present], index=pd.DatetimeIndex(t[present])).sort_index()
    me_s = pd.Series(Y[me], index=pd.DatetimeIndex(t[me])).sort_index()
    win_all = max(20, min(250, len(all_s) // 20))
    win_me = max(8, min(60, round(len(me_s) / 8)))
    fig, ax = plt.subplots(figsize=(13, 3.6))
    ax.scatter(me_s.index, me_s.values, s=6, color=C['bad'], alpha=0.25, lw=0, label='이 경로 웨이퍼')
    ax.plot(all_s.rolling(win_all).mean(), color=C['muted'], lw=2, label=f'스텝 전체 이동평균 ({win_all}장)')
    ax.plot(me_s.rolling(win_me).mean(), color=C['bad'], lw=2.5, label=f'이 경로 이동평균 ({win_me}장)')
    ax.axhline(res['mu'], color=C['muted'], lw=1, ls=(0, (4, 3)))
    rm = pd.concat([all_s.rolling(win_all).mean(), me_s.rolling(win_me).mean()]).dropna()
    if len(rm):
        pad = (rm.max() - rm.min()) * 0.3 + 1e-9
        ax.set_ylim(min(rm.min(), res['mu']) - pad, rm.max() + pad)
    ax.set(title=f"시간 추이 · {steps[s]['name']} track-in 시각", ylabel=Y_NAME)
    ax.legend(loc='lower right', bbox_to_anchor=(1, 1.0), ncol=3, borderaxespad=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

## 4. 조합 판정과 자동 점검
판정 전에 세 가지를 자동으로 점검하고, 필요하면 알아서 보정합니다. 설정을 따로 바꿀 필요는 없습니다. 판정 기준은 하나입니다: y_value와 bad 비율을 합친 결합 z가 오더 수별 조합 수로 보정한 기준 z를 넘는가.

**① 분포 모양**: `y_value`가 한쪽으로 길게 치우치면(왜도 1 초과) 조합 평균이 몇 장의 아주 큰 값에 흔들립니다. 웨이퍼가 적은 조합은 큰 값 두세 장만으로 기준을 넘고, 웨이퍼가 많은 조합은 진짜 차이가 꼬리에 묻힙니다. 이때(`Y_TRANSFORM = 'auto'`) 값을 **순위 점수**로 바꿔 분석합니다. 순위 점수는 웨이퍼를 값 순서로 줄 세운 뒤 그 순위를 정규분포 점수로 옮긴 값이라, 순서는 그대로이고 아주 큰 값 한 장이 평균을 끌고 가지 못합니다. 표에는 원래 `y_value` 평균도 함께 적습니다.

**② part별 수준 차이**: 여러 part가 같은 스텝을 지나므로 함께 분석합니다. 그런데 part마다 `y_value` 수준이나 bad 비율이 다르고 특정 유닛이 특정 part에 몰리면, part 차이가 유닛 불량처럼 보입니다. y_value와 bad 비율 각각 part 간 차이가 뚜렷하면(`PART_ADJUST = 'auto'`) 웨이퍼 값에서 그 part의 평균을 빼고 전체 평균을 더해 수준을 맞춘 뒤 분석합니다.

**③ 랏 차이와 결합 z**: 같은 랏의 웨이퍼는 같은 이력을 거쳐 값이 서로 비슷합니다(랏 간 편차 비율 ICC). 웨이퍼끼리 독립이라고 보면, 랏이 통째로 지나간 유닛은 웨이퍼 수만큼 증거가 있는 것처럼 보여 z가 부풀려집니다. 그래서 y_value와 bad 비율 각각 z를 **랏 랜덤 효과 모형**으로 구합니다.
- 한 랏의 웨이퍼가 여러 챔버로 나뉘면 같은 랏 안에서 비교하므로 랏 차이가 상쇄됩니다.
- 랏이 통째로 한 설비를 지나면 랏 단위로 비교합니다. 랏마다 5장씩 20랏이 지난 설비는 웨이퍼 100장이 아니라 사실상 랏 20개만큼의 증거입니다.
- 비교가 가진 정보량을 **유효 웨이퍼 수**라고 부르고 funnel의 가로축으로 씁니다. ICC가 0에 가까우면 이전(웨이퍼끼리 독립)과 같은 계산이 됩니다.
- bad 비율은 bad가 드물고 웨이퍼가 적으면 정규 근사 z가 실제보다 커지므로, 정확한 이항 꼬리확률로 z를 줄여 맞춥니다. 예: 웨이퍼 25장 중 bad 13장(전체 bad 비율 16%)은 정규 근사 z 4.96, 정확한 이항 꼬리확률로는 4.01.
- 두 z를 하나로 합칩니다: **결합 z = (z_y + z_bad) ÷ √(2 + 2r)**. r은 두 z가 우연히 같이 움직이는 정도(상관)라 같은 정보를 두 번 세지 않게 해 줍니다. y_value와 bad 비율이 함께 나쁘면 크게 오르고, 한쪽만 나쁘면 덜 오릅니다. good_bad가 없으면 y_value z만 씁니다.

In [ ]:
# ① 분포 모양: y_value가 한쪽으로 길게 치우치면 순위 점수(순위를 정규분포 점수로 옮긴 값)로 바꾼다. 순서는 그대로
skew = float(pd.Series(data['Y']).skew())
tf = ('rank' if abs(skew) > 1 else None) if Y_TRANSFORM == 'auto' else Y_TRANSFORM
assert tf in (None, 'rank', 'log'), f"Y_TRANSFORM은 'auto', 'rank', 'log', None 중 하나: {Y_TRANSFORM}"
if tf == 'rank':
    data['Y'] = -norm_isf((pd.Series(data['Y']).rank().to_numpy() - 0.5) / N)
    Y_NAME += ' 순위 점수'
elif tf == 'log':
    lo = data['Y'].min()
    data['Y'] = np.log(data['Y'] if lo > 0 else data['Y'] - lo + max(1e-9, 0.01 * float(np.median(data['Y'] - lo))))
    Y_NAME = f'log {Y_NAME}'
Y = data['Y']
tf_note = {'rank': ' · 순위 점수', 'log': ' · 로그'}.get(tf, '')
print(f'분포: y_value 왜도 {skew:.2f} → ' + {'rank': '순위 점수로 바꿔 분석합니다', 'log': '로그로 바꿔 분석합니다'}.get(tf, '값 그대로 분석합니다'))

# ② part별 수준 차이: y_value와 bad 비율 각각, part 간 차이가 뚜렷하면 웨이퍼 값에서 part 평균을 빼고 전체 평균을 더한다
part_note = ''
if len(set(data['parts'])) > 1 and PART_ADJUST is not False:
    checks, table = [], {}
    for key_, label in (('Y', Y_NAME), ('B', 'bad 비율')):
        v = data[key_]
        if v is None:
            continue
        pg = pd.DataFrame({'part': data['parts'], 'v': v}).groupby('part')['v']
        n_p, m_p = pg.size(), pg.mean()
        eta2 = float((n_p * (m_p - v.mean()) ** 2).sum()) / float(((v - v.mean()) ** 2).sum())
        an_p = anova(n_p.to_numpy(), pg.sum().to_numpy(), pg.apply(lambda x: (x * x).sum()).to_numpy())
        p_part = an_p['p'] if an_p else 1.0
        table['웨이퍼'], table[f'평균 {label}'] = n_p, m_p
        checks.append((key_, label, eta2, p_part, PART_ADJUST is True or (p_part < 0.001 and eta2 >= 0.01), pg))
    display(pd.DataFrame(table).sort_values('웨이퍼', ascending=False).head(10).rename_axis('part_id'))
    done = []
    for key_, label, eta2, p_part, adjust, pg in checks:
        print(f'part 간 {label} 차이: 설명 비율 {eta2:.1%} · p {fmt_p(p_part)} → ' + ('part 평균을 맞춰 분석합니다' if adjust else '보정하지 않습니다'))
        if adjust:
            data[key_] = data[key_] - pg.transform('mean').to_numpy() + data[key_].mean()
            done.append('y_value' if key_ == 'Y' else 'bad')
    if done:
        part_note = f" · part 보정({', '.join(done)})"
        Y_NAME += ' (part 보정)' if 'y_value' in done else ''
    Y = data['Y']

# ③ 랏 차이와 결합 z: 신호(y_value, bad)마다 랏 랜덤 효과 모형으로 z를 구해 하나로 합친다
t0 = time.perf_counter()
res = analyze(data, min_n=MIN_N, max_depth=MAX_DEPTH)
print(f"조합 {len(res['combos']):,}개 (오더 {'/'.join(map(str, range(1, MAX_DEPTH + 1)))}개: {' / '.join(f'{v:,}' for v in res['depth_count'][1:])}) · "
      f"탐색 공간 {' / '.join(f'{v:,}' for v in res['search_space'][1:])} · {time.perf_counter() - t0:.1f}초")
print(f"랏 간 편차 비율(ICC) y_value {res['rho']:.1%}" + (f" · bad {res['rho_b']:.1%}" if data['B'] is not None else '')
      + " → 랏 랜덤 효과 모형으로 z를 구합니다 (같은 랏 안의 비교는 랏 차이가 상쇄되고, 랏이 통째로 지난 유닛은 랏 단위로 비교)")
if data['B'] is not None:
    print(f"판정 z = (y_value z + bad z) ÷ √(2 + 2r) · 두 z가 우연히 같이 움직이는 정도(상관) r = {res['r']:.2f}")
zk = thresholds(res)
ratio_k = defaultdict(list)
for c in res['combos']:
    ratio_k[c['k']].append(c['n_eff'] / c['n'])
display(pd.DataFrame({'기준 z': zk[1:], '유효 웨이퍼 ÷ 웨이퍼 (중앙값)': [float(np.median(ratio_k[k])) if ratio_k[k] else math.nan
                                                               for k in range(1, MAX_DEPTH + 1)]},
                     index=pd.Index(range(1, MAX_DEPTH + 1), name='오더 수')))
summary = classify(res, zk, data)
build_index(res)
print(f"판정 기준: 조합 수 보정 · 랏 랜덤 효과{' · y_value와 bad 결합' if data['B'] is not None else ''}{tf_note}{part_note}")
print(f"판정: 기준 밖 {summary['over']:,}개 → 불량 {summary['bad']}개 · 관련 조합(상속 {summary['inherited']}, 하위 기인 {summary['explained']})")

## 5. 한 장의 funnel과 대표 불량 대상 랭킹
모든 스텝 · 모든 조합 크기를 한 장에 그립니다. 가로축은 유효 웨이퍼 수(랏 차이를 뺀 비교가 가진 정보량), 띠는 우연 범위입니다. 우연 범위 밖의 점은 같은 스텝에서 유닛을 공유하는 **대표 불량 대상**으로 모아, 초과 bad 웨이퍼가 많은 순으로 랭킹을 매깁니다.
- **유효 웨이퍼**: 랏 차이를 뺀 비교가 가진 정보량(웨이퍼 장수 단위). 웨이퍼 수보다 많이 작으면 랏이 통째로 지나간 조합이라 랏 단위 증거만 있음
- **초과 bad 웨이퍼**: 웨이퍼 × (bad 비율 − 전체 bad 비율). 이 대상을 고치면 줄어드는 bad 웨이퍼 수로, 랭킹 순서와 대표 선택의 기준입니다. good_bad가 없으면 웨이퍼 × y_value 차이
- **z ÷ 기준 z**: 결합 z가 기준을 몇 배 넘었는지. 우연이 아닐 확실성
- **z y_value · z bad**: 결합 전 두 z. 어느 쪽이 판정을 끌었는지 봅니다
- **차이**: 분석한 y_value(순위 점수 등) 평균의 차이. 웨이퍼 한 장당 심각도
- **랏 수**: 조합의 웨이퍼가 속한 랏 수. 적으면 몇 개 랏의 우연일 수 있으니 조심해서 봄
- **같은 웨이퍼 대상**: 웨이퍼가 `SAME_WAFERS`(90%) 이상 겹치는 대표 대상. 예를 들어 하위 스텝 `_1`, `_2`, `_3`을 같은 챔버에서 진행하면 세 오더의 웨이퍼가 같아서 데이터로는 어느 오더 탓인지 가를 수 없으므로 한 줄로 묶습니다. 웨이퍼가 일부만 겹치는 대상은 따로 한 줄씩 나옵니다.
- **스텝 간 판정**: 다른 스텝의 불량이 옮겨 온 것으로 보이면 '다른 스텝 기인'

In [ ]:
over_pts = [c for c in res['combos'] if c['over'] and in_scope(c)]
by_target = defaultdict(list)
for c in over_pts:
    t = representative(res, zk, c)
    by_target[t['key'] if t else None].append(c)
cross = cross_step_check(data, res, zk)

rep_rows = []
for key, pts in by_target.items():
    if key is not None:
        c = res['by_key'][key]
        a = cross.get(key)
        rep_rows.append({**combo_row(data, res, zk, c), '연결된 점': len(pts),
                         '스텝 간 판정': f"다른 스텝 기인 ({steps[a['step']]['name']} {path_label(steps[a['step']], a['items'])})" if a else '–',
                         'key': key})
rep_df = pd.DataFrame(rep_rows)
same_as = {}                     # 한 줄로 묶인 대상 key → 남긴 대상 key (웨이퍼가 거의 같은 대상)
if len(rep_df):
    rep_df = rep_df.sort_values(RANK_NAME, ascending=False, kind='stable')
    kept = []
    for key in rep_df['key']:
        m = members_of(data, res['by_key'][key])
        hit = next((k2 for k2, m2 in kept if SAME_WAFERS and (m & m2).sum() >= SAME_WAFERS * (m | m2).sum()), None)
        if hit is None:
            kept.append((key, m))
        else:
            same_as[key] = hit


def same_text(key):
    """한 줄로 묶인 대상의 경로. 같은 스텝이면 경로만, 다른 스텝이면 스텝과 경로"""
    s0 = res['by_key'][key]['step']
    return ', '.join(path_label(steps[b['step']], b['items']) if b['step'] == s0 else f"{steps[b['step']]['name']} {path_label(steps[b['step']], b['items'])}"
                     for b in (res['by_key'][k] for k, v in same_as.items() if v == key))


n_shown = sum(in_scope(c) for c in res['combos'])
print(f"우연 범위 밖의 점 {len(over_pts):,}개 → 대표 불량 대상 {len(rep_df) - len(same_as)}개"
      + (f" (웨이퍼가 같은 대상 {len(same_as)}개를 한 줄로 묶음)" if same_as else '')
      + (f" · 확정 불량 없음 {len(by_target[None])}개" if None in by_target else ''))
fig, ax = plt.subplots(figsize=(13, 5.2))
plot_unified(ax, data, res, zk, cross=cross, skip_label=same_as, title=f'모든 스텝 · 모든 조합 크기 {n_shown:,}개 · 우연 범위 밖 {len(over_pts):,}개')
plt.tight_layout()
plt.show()

if len(rep_df):
    n_pts = dict(zip(rep_df['key'], rep_df['연결된 점']))
    rep_df = rep_df[~rep_df['key'].isin(same_as)].copy()
    rep_df['연결된 점'] = [n_pts[k] + sum(n_pts[a] for a, v in same_as.items() if v == k) for k in rep_df['key']]
    rep_df.insert(rep_df.columns.get_loc('경로') + 1, '같은 웨이퍼 대상', [same_text(k) or '–' for k in rep_df['key']])
    rep_df.index = pd.RangeIndex(1, len(rep_df) + 1, name=f'순위 ({RANK_NAME})')
    display(rep_df)
    if SAVE_RANKING:
        rep_df.to_csv(SAVE_RANKING, encoding='utf-8-sig')
        print(f'랭킹 표 저장: {SAVE_RANKING}')
else:
    print('대표 불량 대상이 없습니다.')

## 6. 대표 불량 대상 자세히 보기
랭킹 `PICK`위 대상을 봅니다.
- **후보 표**: 같은 스텝에서 기준을 넘은 관련 조합과, 각 후보가 대표가 아닌 이유
- **스텝 흐름 · 웨이퍼 분포 · 왜 대표인가 · 시간 추이**

In [ ]:
if len(rep_df):
    T = res['by_key'][rep_df.iloc[PICK - 1]['key']]
    st = steps[T['step']]
    print(f"대표 불량 대상 {PICK}위: {st['name']} · {path_label(st, T['items'])} · 웨이퍼 {T['n']:,}장 · "
          f"평균 {Y_NAME} {T['mean']:.4g} (전체 {round(res['mu'], 6) + 0:.4g}, 차이 {T['mean'] - res['mu']:+.4g})"
          + (f" · 원래 y_value 평균 {np.nanmean(data['value'][members_of(data, T)]):.4g} (전체 {np.nanmean(data['value']):.4g})"
             if Y_NAME != 'y_value' else '')
          + (f" · bad 비율 {np.nanmean(data['bad'][members_of(data, T)]):.1%} (전체 {np.nanmean(data['bad']):.1%})"
             if data['B'] is not None else ''))
    print(f"판정 z {T['z']:.2f} (기준 {zk[T['k']]:.2f})" + (f" · y_value z {T['z_y']:.2f} · bad z {T['z_b']:.2f}" if data['B'] is not None else ''))
    print('경로의 오더: ' + ' · '.join(f"{st['seqs'][q]['name']} = {st['seqs'][q]['seq']} ({st['seqs'][q]['desc']})" for q, _ in T['items']))
    if same_text(T['key']):
        print(f"같은 웨이퍼 대상: {same_text(T['key'])} (웨이퍼가 거의 같아서 데이터로는 어느 쪽 탓인지 가를 수 없음)")
    cands = sorted(candidates(res, T), key=lambda b: -metric(b, res))
    display(pd.DataFrame([{**combo_row(data, res, zk, b), '판정': reason(res, zk, st, b, T)} for b in cands],
                         index=pd.RangeIndex(1, len(cands) + 1, name='후보')).drop(columns=['스텝', 'step_desc']))

In [ ]:
if len(rep_df):
    plot_flow(T)
    me, others, parents = membership(T)
    fig, ax = plt.subplots(figsize=(12, 1.6 + 0.62 * (T['k'] + 2 if T['k'] >= 2 else 2)))
    plot_dist(ax, T, me, others, parents)
    plt.tight_layout()
    plt.show()

**왜 대표 불량 대상인가**: ●는 조합 전체, ○는 비교할 웨이퍼, 띠는 95% 신뢰구간입니다.
- **②** 관련 조합에서 대표 대상의 웨이퍼를 빼면(● → ○) 전체 평균으로 돌아갑니다. 관련 조합이 높았던 것은 대표 대상의 웨이퍼 때문입니다.
- **③** 대표 대상에 다른 오더의 유닛을 더해 좁힌 조합(●)이 대표 대상의 나머지 웨이퍼(○)와 같은 수준입니다. 더 좁혀도 더 나빠지지 않으므로, 더 구체적인 경로가 원인이 아닙니다.

In [ ]:
if len(rep_df):
    plot_why(T)
    plot_time(T)

## 7. 스텝 간 설비 연관 점검
판정은 스텝마다 따로 하므로, 두 스텝의 설비가 짝지어 운영되면 한 원인이 여러 스텝의 불량 조합으로 보일 수 있습니다. 불량 조합끼리 웨이퍼가 겹치면, 상대 스텝의 웨이퍼를 뺀 나머지로 효과가 남는지 봅니다.

In [ ]:
if cross:
    display(pd.DataFrame([{
        '불량 조합': f"{steps[res['by_key'][k]['step']]['name']} {path_label(steps[res['by_key'][k]['step']], res['by_key'][k]['items'])}",
        '원인으로 보이는 조합': f"{steps[a['step']]['name']} {path_label(steps[a['step']], a['items'])}",
    } for k, a in cross.items()], index=pd.RangeIndex(1, len(cross) + 1, name='다른 스텝 기인')))
else:
    print('다른 스텝의 불량이 옮겨 온 것으로 보이는 불량 조합이 없습니다.')